# Algebraic interaction analysis on SkinCon: baseline models (replication family)

Experiment notebook of *Algebraic Interaction Analysis for Concept-Based Explanations in Skin Cancer Diagnosis* (Nápoles and Grau). This is the notebook that produced every result of the baseline models reported in the paper (output folder `AIA_outputs/`). Relative to the executed version, only this introduction cell and the checkpoint folder (`MODELS_PATH = .../models_baseline`, so that the checkpoints of the two families never overwrite each other) were changed.

**Part 1** uses the same data, duplicate groups and frozen split file as the tuned models. It trains one EfficientNet-B3 concept predictor per seed (42, 64, 128) with the settings of the published Derm7pt concept bottleneck model: the first two of nine feature blocks frozen, Adam with a learning rate of 5e-4 and a cosine decay to 1e-5 over 20 epochs, batches of 16 images at 224 pixels, class-weighted cross-entropy and RandAugment of magnitude 9. The epoch with the largest validation concept macro F1 score is kept, and hard states are the argmax of each concept head. The diagnosis heads are the same five heads as for the tuned models.

**Part 2** trains nothing and never opens an image. It runs the four experiments of the paper on the predicted concept states and the diagnosis heads of Part 1: computation of local ranking laws (identities, samplers against exact laws for 12, 16 and 20 concepts, Monte Carlo expected-hard games), interactions (Walsh analysis, parity duality, certificates with Hoeffding and empirical Bernstein radii, decision diagnostics, pair indices of other methods), compact summaries (exact maximum-entropy fits of the soft ranking and of the average positions) and persistence across retrained models and removal rules.

The inverse temperature and the model studied in depth are frozen on validation images. The console is mirrored to `console_log.txt`, every table is written to one csv file per table type in `csv/`, and the last cell writes a zip archive of the output folder. Expensive steps are cached in `cache/`, so a restarted run resumes where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# ============================================================
# GLOBAL SETTINGS SHARED BY BOTH PARTS
# ============================================================
SEEDS = [42, 64, 128, 256, 512]
BACKBONE_NAME = "efficientnet_b3"
USE_DDI = True                 # False uses the Fitzpatrick17k part of SkinCon only
DOWNLOAD_FITZ_IMAGES = False   # True tries to download missing Fitzpatrick17k images from the metadata URLs
RUN_TRAINING = True            # False reuses the checkpoints of an earlier run
SKIP_TRAINED_SEEDS = True

PROJECT_DIR = "/content/drive/MyDrive/Notebooks/SkinCon"
SKINCON_DIR = f"{PROJECT_DIR}/skincon"      # annotations_fitzpatrick17k.csv, annotations_ddi.csv
FITZ_META = f"{PROJECT_DIR}/fitzpatrick17k/fitzpatrick17k.csv"   # metadata of github.com/mattgroh/fitzpatrick17k
FITZ_IMAGES = f"{PROJECT_DIR}/fitzpatrick17k/images"             # files named <md5hash>.jpg
DDI_DIR = f"{PROJECT_DIR}/ddi"                                    # ddi_metadata.csv plus the DDI image files
MODELS_PATH = f"{PROJECT_DIR}/models_baseline"   # separate folder: the tuned notebook writes checkpoints with the same file names
AIA_OUTPUT_PATH = f"{PROJECT_DIR}/AIA_outputs"

# Part 1. Data, duplicate groups, splits and sequential concept bottleneck models

Download the two SkinCon annotation files from https://skincon-dataset.github.io, the Fitzpatrick17k metadata and images from https://github.com/mattgroh/fitzpatrick17k, and (optionally) DDI from https://ddi-dataset.github.io. Images flagged "Do not consider this image" are removed. The diagnosis target is malignant versus non-malignant (three-partition label of Fitzpatrick17k, malignant flag of DDI).

Fitzpatrick17k contains near-duplicate images, so images whose perceptual hashes differ in at most `PHASH_MAX_BITS` bits (or whose bytes coincide) form one group, and a stratified group split keeps every group inside one split. The split does not depend on the model seed.

In [ ]:
# ============================================================
# DATA. SKINCON ANNOTATIONS, IMAGES, DUPLICATE GROUPS, SPLITS AND CONCEPT VOCABULARIES
# The split file is written once and reused by every later run, so every seed, head and experiment sees the
# same training, validation and test images. Delete SPLIT_FILE to rebuild it.
# ============================================================
import os
import io
import sys
import json
import time
import math
import random
import hashlib
import warnings
import builtins

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")
try:
    import imagehash
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash"], check=True)
    import imagehash
from sklearn.model_selection import StratifiedGroupKFold

original_print = builtins.print
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

SPLIT_SEED = 2024                 # fixed, independent of the model seeds
PHASH_MAX_BITS = 6                # perceptual hashes within this Hamming distance join one duplicate group
MIN_CONCEPT_IMAGES = 50           # concepts with fewer positive images in SkinCon are never used
VOCAB_SIZES = (12, 16, 20)        # nested vocabularies of the most frequent training concepts
SPLIT_DIR = os.path.join(PROJECT_DIR, "splits")
SPLIT_FILE = os.path.join(SPLIT_DIR, f"skincon_splits_{'fitz_ddi' if USE_DDI else 'fitz'}.csv")
VOCAB_FILE = SPLIT_FILE.replace(".csv", "_vocabularies.json")
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

LOCAL_IMAGE_DIR = "/content/skincon_images"   # fast local copy of every image (Google Drive is slow per file)
IO_THREADS = 32                                # parallel reads from Drive and parallel hashing

EXCLUDE_COLUMN = "Do not consider this image"
ANNOT_SKIP = {"Unnamed: 0", "ImageID", EXCLUDE_COLUMN}


def concept_key(name):
    """Column-safe concept key, e.g. 'Brown(Hyperpigmentation)' -> 'brown_hyperpigmentation'."""
    k = "".join(ch.lower() if ch.isalnum() else "_" for ch in name)
    while "__" in k:
        k = k.replace("__", "_")
    return k.strip("_")


def download_fitz_images(df, target_dir, timeout=20):
    """Best-effort download of missing Fitzpatrick17k images from the metadata URLs (many URLs no longer work)."""
    import requests
    os.makedirs(target_dir, exist_ok=True)
    ok = fail = 0
    for _, r in df.iterrows():
        dst = os.path.join(target_dir, r["ImageID"])
        if os.path.exists(dst):
            continue
        try:
            resp = requests.get(r["url"], timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
            resp.raise_for_status()
            Image.open(io.BytesIO(resp.content)).convert("RGB").save(dst, quality=95)
            ok += 1
        except Exception:
            fail += 1
    original_print(f"Fitzpatrick17k download: {ok} new images, {fail} failures")


def stage_images(files_by_source):
    """
    Copy every image once from Google Drive to the local disk, with parallel threads and progress output.
    Reading thousands of small files from Drive one at a time takes hours, local reads take milliseconds.
    Files already present locally are skipped, so a restarted session only copies what is missing.
    A zip archive <folder>.zip next to an image folder is extracted instead, which is faster still.
    """
    import shutil
    import zipfile
    from concurrent.futures import ThreadPoolExecutor
    sources = {"fitzpatrick17k": FITZ_IMAGES, "ddi": DDI_DIR}
    found = {}
    for src, files in files_by_source.items():
        dst_dir = os.path.join(LOCAL_IMAGE_DIR, src)
        os.makedirs(dst_dir, exist_ok=True)
        zpath = sources[src].rstrip("/") + ".zip"
        todo = [f for f in files if not os.path.exists(os.path.join(dst_dir, f))]
        if todo and os.path.exists(zpath):
            t0 = time.time()
            with zipfile.ZipFile(zpath) as zf:
                names = {os.path.basename(n): n for n in zf.namelist() if not n.endswith("/")}
                for f in todo:
                    if f in names:
                        with zf.open(names[f]) as a, open(os.path.join(dst_dir, f), "wb") as b:
                            shutil.copyfileobj(a, b)
            original_print(f"  {src}: extracted from {zpath} in {time.time() - t0:.0f} s")
            todo = [f for f in files if not os.path.exists(os.path.join(dst_dir, f))]
        if todo:
            try:
                on_drive = set(os.listdir(sources[src]))           # one directory listing instead of one call per file
            except FileNotFoundError:
                on_drive = set()
            copy = [f for f in todo if f in on_drive]
            t0, done = time.time(), [0]

            def one(f):
                shutil.copyfile(os.path.join(sources[src], f), os.path.join(dst_dir, f))
                done[0] += 1
                if done[0] % 250 == 0 or done[0] == len(copy):
                    el = time.time() - t0
                    original_print(f"  {src}: {done[0]}/{len(copy)} images copied, {el:.0f} s elapsed, "
                                   f"about {el / done[0] * (len(copy) - done[0]):.0f} s left", flush=True)

            original_print(f"  {src}: copying {len(copy)} images from Drive with {IO_THREADS} threads", flush=True)
            with ThreadPoolExecutor(IO_THREADS) as ex:
                list(ex.map(one, copy))
        local = set(os.listdir(dst_dir))
        found[src] = {f for f in files if f in local}
        original_print(f"  {src}: {len(found[src])} of {len(files)} images available locally", flush=True)
    return found


def load_skincon():
    """One row per usable SkinCon image with the binary target, the 48 concepts, the source and the image path."""
    fa = pd.read_csv(os.path.join(SKINCON_DIR, "annotations_fitzpatrick17k.csv"))
    concepts = [c for c in fa.columns if c not in ANNOT_SKIP]
    meta = pd.read_csv(FITZ_META)
    fa["md5hash"] = fa["ImageID"].astype(str).str.replace(".jpg", "", regex=False)
    f = fa.merge(meta[["md5hash", "three_partition_label", "label", "fitzpatrick_scale", "qc", "url"]],
                 on="md5hash", how="left", validate="one_to_one")
    counts = {"fitzpatrick17k annotated": len(f)}
    f = f[f[EXCLUDE_COLUMN].fillna(0) == 0]
    counts["fitzpatrick17k after exclusion flag"] = len(f)
    f = f[~f["qc"].astype(str).str.startswith("3")]            # quality control: wrongly labelled
    counts["fitzpatrick17k after quality control"] = len(f)
    f = f[f["three_partition_label"].notna()]
    if DOWNLOAD_FITZ_IMAGES:
        download_fitz_images(f, FITZ_IMAGES)
    f = pd.DataFrame({"image_id": "fitz_" + f["md5hash"], "source": "fitzpatrick17k",
                      "file": f["ImageID"].astype(str).values,
                      "malignant": (f["three_partition_label"] == "malignant").astype(int).values,
                      "diagnosis": f["label"].values, "skin_type": f["fitzpatrick_scale"].astype(str).values,
                      **{concept_key(c): f[c].fillna(0).astype(int).values for c in concepts}})
    frames = [f]
    if USE_DDI:
        da = pd.read_csv(os.path.join(SKINCON_DIR, "annotations_ddi.csv"))
        dm = pd.read_csv(os.path.join(DDI_DIR, "ddi_metadata.csv"))
        d = da.merge(dm, left_on="ImageID", right_on="DDI_file", how="left", validate="one_to_one")
        counts["ddi annotated"] = len(d)
        d = d[(d[EXCLUDE_COLUMN].fillna(0) == 0) & d["malignant"].notna()]
        counts["ddi after exclusion flag"] = len(d)
        mal = d["malignant"].astype(str).str.lower().isin(["true", "1", "1.0", "yes"]).astype(int)
        d = pd.DataFrame({"image_id": "ddi_" + d["ImageID"].astype(str), "source": "ddi",
                          "file": d["ImageID"].astype(str).values,
                          "malignant": mal.values, "diagnosis": d["disease"].values,
                          "skin_type": d["skin_tone"].astype(str).values,
                          **{concept_key(c): d[c].fillna(0).astype(int).values for c in concepts}})
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    original_print("Staging images on the local disk", flush=True)
    found = stage_images({s: df.loc[df["source"] == s, "file"].tolist() for s in df["source"].unique()})
    df["path"] = [os.path.join(LOCAL_IMAGE_DIR, s, f) for s, f in zip(df["source"], df["file"])]
    exists = pd.Series([f in found[s] for s, f in zip(df["source"], df["file"])], index=df.index)
    counts["images found on disk"] = int(exists.sum())
    if (~exists).any():
        original_print(f"Missing image files: {int((~exists).sum())} (listed in missing_images.csv, dropped)")
        df[~exists][["image_id", "source", "file"]].to_csv(os.path.join(SPLIT_DIR, "missing_images.csv"), index=False)
    df = df[exists].reset_index(drop=True)
    return df, [concept_key(c) for c in concepts], dict(zip([concept_key(c) for c in concepts], concepts)), counts


def duplicate_groups(paths, max_bits):
    """Union-find over identical files and perceptual hashes within max_bits Hamming distance."""
    from concurrent.futures import ThreadPoolExecutor
    n = len(paths)
    t0 = time.time()

    def one(p):
        with open(p, "rb") as fh:                 # a single read gives both the byte hash and the image
            raw = fh.read()
        with Image.open(io.BytesIO(raw)) as im:
            h = int(str(imagehash.phash(im.convert("RGB"))), 16)
        return h, hashlib.md5(raw).hexdigest()

    out = []
    with ThreadPoolExecutor(IO_THREADS) as ex:
        for k, r in enumerate(ex.map(one, paths), 1):
            out.append(r)
            if k % 500 == 0 or k == n:
                original_print(f"  hashed {k}/{n} images, {time.time() - t0:.0f} s", flush=True)
    ph = np.array([h for h, _ in out], dtype=np.uint64)
    md5 = [m for _, m in out]
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)

    first = {}
    for i, h in enumerate(md5):
        if h in first:
            union(i, first[h])
        else:
            first[h] = i
    bytes_ = ph.view(np.uint8).reshape(n, 8)
    pairs = 0
    for i in range(0, n, 256):
        x = np.bitwise_xor(bytes_[i:i + 256, None, :], bytes_[None, :, :])        # (b, n, 8)
        dist = np.unpackbits(x, axis=-1).sum(-1)
        for a, b in zip(*np.where(dist <= max_bits)):
            a = a + i
            if a < b:
                union(a, b)
                pairs += 1
    return np.array([find(i) for i in range(n)]), pairs


def build_splits():
    df, all_concepts, concept_names, counts = load_skincon()
    t0 = time.time()
    groups, pairs = duplicate_groups(df["path"].tolist(), PHASH_MAX_BITS)
    df["group"] = groups
    sizes = df.groupby("group").size()
    counts["near-duplicate pairs"] = pairs
    counts["duplicate groups with more than one image"] = int((sizes > 1).sum())
    counts["images in such groups"] = int(sizes[sizes > 1].sum())
    mixed = df.groupby("group")["malignant"].nunique()
    counts["groups with conflicting targets"] = int((mixed > 1).sum())
    original_print(f"Perceptual hashing of {len(df)} images: {time.time() - t0:.0f} s")
    strat = df["malignant"].astype(str) + "_" + df["source"]
    sgk = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
    folds = np.zeros(len(df), dtype=int)
    for k, (_, idx) in enumerate(sgk.split(df, strat, df["group"])):
        folds[idx] = k
    df["split"] = np.where(folds == 0, "test", np.where(folds == 1, "valid", "train"))
    # concept vocabularies from the training split only (ties broken by the SkinCon total, then by name)
    tr = df[df["split"] == "train"]
    eligible = [c for c in all_concepts if df[c].sum() >= MIN_CONCEPT_IMAGES]
    order = sorted(eligible, key=lambda c: (-int(tr[c].sum()), -int(df[c].sum()), c))
    if len(order) < max(VOCAB_SIZES):
        raise ValueError(f"Only {len(order)} concepts have at least {MIN_CONCEPT_IMAGES} images.")
    vocab = {f"V{k}": order[:k] for k in VOCAB_SIZES}
    info = {"vocabularies": vocab, "concept_names": {c: concept_names[c] for c in order},
            "training_counts": {c: int(tr[c].sum()) for c in order}, "eligible": eligible,
            "counts": counts, "split_seed": SPLIT_SEED, "phash_max_bits": PHASH_MAX_BITS, "use_ddi": USE_DDI}
    keep = ["image_id", "source", "file", "malignant", "diagnosis", "skin_type", "group", "split"] + all_concepts
    df[keep].to_csv(SPLIT_FILE, index=False)
    with open(VOCAB_FILE, "w") as fh:
        json.dump(info, fh, indent=1)
    return df[keep + ["path"]], info


if os.path.exists(SPLIT_FILE) and os.path.exists(VOCAB_FILE):
    DATA = pd.read_csv(SPLIT_FILE)
    with open(VOCAB_FILE) as fh:
        VOCAB_INFO = json.load(fh)
    original_print(f"Reusing the frozen split file {SPLIT_FILE}")
    original_print("Staging images on the local disk (a new Colab session starts with an empty disk)", flush=True)
    _found = stage_images({s: DATA.loc[DATA["source"] == s, "file"].tolist() for s in DATA["source"].unique()})
    _miss = sum(f not in _found[s] for s, f in zip(DATA["source"], DATA["file"]))
    if _miss:
        raise FileNotFoundError(f"{_miss} images of the frozen split are missing on Drive.")
    DATA["path"] = [os.path.join(LOCAL_IMAGE_DIR, s, f) for s, f in zip(DATA["source"], DATA["file"])]
else:
    DATA, VOCAB_INFO = build_splits()

VOCABS = VOCAB_INFO["vocabularies"]
CONCEPTS = VOCABS[f"V{max(VOCAB_SIZES)}"]           # concepts predicted by the concept model (largest vocabulary)
CONCEPT_NAMES = VOCAB_INFO["concept_names"]

original_print("\nData assembly")
for k, v in VOCAB_INFO["counts"].items():
    original_print(f"  {k:<45} {v}")
tab = DATA.groupby(["split", "source", "malignant"]).size().unstack(fill_value=0)
original_print("\nImages per split, source and target (1 = malignant)\n" + tab.to_string())
original_print("\nConcept vocabularies (nested, ordered by training frequency)")
for c in CONCEPTS:
    member = [k for k in VOCABS if c in VOCABS[k]]
    original_print(f"  {CONCEPT_NAMES[c]:<28} train={VOCAB_INFO['training_counts'][c]:>5}   "
                   f"prevalence={DATA[c].mean():.3f}   in {', '.join(member)}")

In [ ]:
# ============================================================
# SEQUENTIAL HARD CONCEPT BOTTLENECK MODELS
# Stage 1 trains, for every seed, the image encoder with one two-class head per concept of the largest
# vocabulary (the concept block of the published Derm7pt+ model, class-weighted cross-entropy, 20 epochs,
# model selection on the validation macro F1 of the concepts). The predicted states are one-hot blocks
# (absent, present), so a removed concept (zero block) differs from an absent one.
# Stage 2 trains the diagnosis heads on the frozen predicted states of the training images, with early stopping
# on the validation loss. Every head sees the same concept predictor of its seed:
#   V12 affine   negative control, every removal game is exactly additive
#   V12 mlp      the published two-layer head (primary)
#   V12 deep     three hidden layers, positive control
#   V16 mlp, V20 mlp   larger vocabularies for the computational experiments
# Part 2 needs only the files written here (predicted states, probabilities, heads), not the images.
# ============================================================
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix)


class TrainConfig:
    TUNABLE_LAYERS = 75
    BATCH_SIZE = 16
    EPOCHS = 20
    IMG_SIZE = 224
    LEARNING_RATE = 5e-4
    ETA_MIN = 1e-5
    CONCEPT_WEIGHTING = True
    HEAD_EPOCHS = 300
    HEAD_PATIENCE = 40
    HEAD_LR = 1e-3
    HEAD_WEIGHT_DECAY = 1e-4
    HEAD_BATCH = 64
    HEAD_HIDDEN = 128
    HEAD_SPECS = [("V12", "affine"), ("V12", "mlp"), ("V12", "deep"), ("V16", "mlp"), ("V20", "mlp")]
    NUM_WORKERS = 2
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    PRETRAINED = True


TCFG = TrainConfig()


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


BACKBONES = {
    "efficientnet_b0": (models.efficientnet_b0, "EfficientNet_B0_Weights", 1280, "features"),
    "efficientnet_b3": (models.efficientnet_b3, "EfficientNet_B3_Weights", 1536, "features"),
    "efficientnet_b4": (models.efficientnet_b4, "EfficientNet_B4_Weights", 1792, "features"),
    "densenet121": (models.densenet121, "DenseNet121_Weights", 1024, "features"),
    "resnet18": (models.resnet18, "ResNet18_Weights", 512, "children"),
    "resnet50": (models.resnet50, "ResNet50_Weights", 2048, "children"),
}


def get_backbone(name, pretrained=True):
    fn, wname, dim, kind = BACKBONES[name]
    m = fn(weights=getattr(models, wname).DEFAULT if pretrained else None)
    features = m.features if kind == "features" else nn.Sequential(*list(m.children())[:-2])
    total = len(features)
    freeze_up_to = int(round(total * (100 - TCFG.TUNABLE_LAYERS) / 100))
    for i, layer in enumerate(features):
        for p in layer.parameters():
            p.requires_grad = i >= freeze_up_to
    return features, dim


class ConceptModel(nn.Module):
    """Image encoder with one two-class head per concept (the concept block of the published model)."""

    def __init__(self, backbone_name, concepts, pretrained=True):
        super().__init__()
        self.features, self.dim = get_backbone(backbone_name, pretrained)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.concepts = list(concepts)
        self.heads = nn.ModuleDict({c: nn.Linear(self.dim, 2) for c in self.concepts})

    def forward(self, x):
        z = self.pool(self.features(x)).flatten(1)
        return [self.heads[c](z) for c in self.concepts]


class SkinConDataset(Dataset):
    def __init__(self, df, concepts, transform):
        self.paths = df["path"].tolist()
        self.c = torch.tensor(df[concepts].values, dtype=torch.long)
        self.y = torch.tensor(df["malignant"].values, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.c[i], self.y[i]


def train_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(TCFG.IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(90),
        transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15)], p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


def eval_transforms():
    return transforms.Compose([
        transforms.Resize((TCFG.IMG_SIZE, TCFG.IMG_SIZE)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


def concept_class_weights(df, concepts):
    out = []
    for c in concepts:
        pos = float(df[c].mean())
        pos = min(max(pos, 1e-3), 1 - 1e-3)
        w = np.array([0.5 / (1 - pos), 0.5 / pos]) if TCFG.CONCEPT_WEIGHTING else np.ones(2)
        out.append(torch.tensor(w, dtype=torch.float32, device=TCFG.DEVICE))
    return out


@torch.no_grad()
def concept_outputs(model, df, concepts):
    """Probability of presence for every concept (float64) and the hard states (argmax of each head)."""
    model.eval()
    dl = DataLoader(SkinConDataset(df, concepts, eval_transforms()), batch_size=32, shuffle=False,
                    num_workers=TCFG.NUM_WORKERS)
    probs = []
    for x, _, _ in dl:
        lg = model(x.to(TCFG.DEVICE))
        probs.append(torch.stack([torch.softmax(l.double(), 1)[:, 1] for l in lg], 1).cpu().numpy())
    p = np.concatenate(probs)
    hard = np.stack([np.argmax(np.stack([1 - p[:, j], p[:, j]], 1), 1) for j in range(p.shape[1])], 1)
    return p, hard


def concept_metrics(hard, df, concepts):
    t = df[concepts].values
    f1 = [f1_score(t[:, j], hard[:, j], average="macro", zero_division=0) for j in range(len(concepts))]
    bacc = [balanced_accuracy_score(t[:, j], hard[:, j]) if len(np.unique(t[:, j])) > 1 else np.nan
            for j in range(len(concepts))]
    return float(np.mean(f1)), float(np.nanmean(bacc)), f1


def train_concept_model(seed, splits):
    path = os.path.join(MODELS_PATH, f"concepts_{BACKBONE_NAME}_seed{seed}.pth")
    if SKIP_TRAINED_SEEDS and os.path.exists(path):
        original_print(f"  seed {seed}: reusing {path}")
        return path
    set_seed(seed)
    model = ConceptModel(BACKBONE_NAME, CONCEPTS, TCFG.PRETRAINED).to(TCFG.DEVICE)
    g = torch.Generator()
    g.manual_seed(seed)
    dl = DataLoader(SkinConDataset(splits["train"], CONCEPTS, train_transforms()), batch_size=TCFG.BATCH_SIZE,
                    shuffle=True, num_workers=TCFG.NUM_WORKERS, generator=g, drop_last=True)
    wts = concept_class_weights(splits["train"], CONCEPTS)
    opt = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=TCFG.LEARNING_RATE)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TCFG.EPOCHS, eta_min=TCFG.ETA_MIN)
    best, best_state, best_epoch, history = -1.0, None, -1, []
    for ep in range(1, TCFG.EPOCHS + 1):
        model.train()
        tot = 0.0
        for x, c, _ in dl:
            x, c = x.to(TCFG.DEVICE), c.to(TCFG.DEVICE)
            opt.zero_grad()
            lg = model(x)
            loss = torch.stack([F.cross_entropy(lg[j], c[:, j], weight=wts[j]) for j in range(len(CONCEPTS))]).mean()
            loss.backward()
            opt.step()
            tot += loss.item()
        sched.step()
        _, hv = concept_outputs(model, splits["valid"], CONCEPTS)
        f1v, baccv, _ = concept_metrics(hv, splits["valid"], CONCEPTS)
        history.append({"epoch": ep, "train_loss": tot / max(len(dl), 1), "valid_concept_f1": f1v,
                        "valid_concept_bacc": baccv})
        original_print(f"  seed {seed} epoch {ep:>2}: loss {tot / max(len(dl), 1):.4f}  "
                       f"valid concept macro F1 {f1v:.4f}  balanced accuracy {baccv:.4f}")
        if f1v > best:
            best, best_epoch, best_state = f1v, ep, copy.deepcopy(model.state_dict())
    torch.save({"state_dict": best_state, "backbone": BACKBONE_NAME, "concepts": CONCEPTS, "seed": seed,
                "best_epoch": best_epoch, "best_valid_concept_f1": best, "history": history}, path)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return path


# ---------------- diagnosis heads on frozen predicted states ----------------

def one_hot_blocks(hard):
    """(L, n) binary states -> (L, 2n) one-hot blocks (absent, present)."""
    L, n = hard.shape
    X = np.zeros((L, 2 * n))
    X[np.arange(L)[:, None], 2 * np.arange(n)[None, :] + hard] = 1.0
    return X


def make_head(kind, n):
    d, h = 2 * n, TCFG.HEAD_HIDDEN
    if kind == "affine":
        return nn.Sequential(nn.Linear(d, 2))
    if kind == "mlp":
        return nn.Sequential(nn.Linear(d, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(0.3), nn.Linear(h, 2))
    if kind == "deep":
        layers, w = [], d
        for _ in range(3):
            layers += [nn.Linear(w, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(0.3)]
            w = h
        return nn.Sequential(*layers, nn.Linear(h, 2))
    raise ValueError(kind)


def head_spec(head):
    """Float64 description of a trained head, read by the numpy mirror of Part 2."""
    spec = []
    for m in head:
        if isinstance(m, nn.Linear):
            spec.append(("linear", m.weight.detach().double().cpu().numpy(), m.bias.detach().double().cpu().numpy()))
        elif isinstance(m, nn.LayerNorm):
            spec.append(("layernorm", m.weight.detach().double().cpu().numpy(), m.bias.detach().double().cpu().numpy(),
                         float(m.eps)))
        elif isinstance(m, nn.ReLU):
            spec.append(("relu",))
    return spec


def label_metrics(y, s):
    pred = (s > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"), "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1), "auroc": roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan}


def train_head(kind, cols, hard, y, seed):
    set_seed(seed)
    n = len(cols)
    Xtr = torch.tensor(one_hot_blocks(hard["train"][:, cols]), dtype=torch.float32)
    ytr = torch.tensor(y["train"], dtype=torch.long)
    Xva = torch.tensor(one_hot_blocks(hard["valid"][:, cols]), dtype=torch.float32)
    yva = torch.tensor(y["valid"], dtype=torch.long)
    cw = torch.tensor(len(ytr) / (2.0 * np.bincount(y["train"], minlength=2).clip(1)), dtype=torch.float32)
    head = make_head(kind, n)
    opt = optim.AdamW(head.parameters(), lr=TCFG.HEAD_LR, weight_decay=TCFG.HEAD_WEIGHT_DECAY)
    g = torch.Generator()
    g.manual_seed(seed)
    best, best_state, best_epoch, wait = np.inf, None, 0, 0
    for ep in range(1, TCFG.HEAD_EPOCHS + 1):
        head.train()
        perm = torch.randperm(len(ytr), generator=g)
        for i in range(0, len(ytr), TCFG.HEAD_BATCH):
            b = perm[i:i + TCFG.HEAD_BATCH]
            opt.zero_grad()
            F.cross_entropy(head(Xtr[b]), ytr[b], weight=cw).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            vl = F.cross_entropy(head(Xva), yva, weight=cw).item()
        if vl < best - 1e-6:
            best, best_state, best_epoch, wait = vl, copy.deepcopy(head.state_dict()), ep, 0
        else:
            wait += 1
            if wait >= TCFG.HEAD_PATIENCE:
                break
    head.load_state_dict(best_state)
    head.eval()
    return head, best_epoch, best


def head_logit_torch(head, hard):
    with torch.no_grad():
        o = head(torch.tensor(one_hot_blocks(hard), dtype=torch.float32))
    return (o[:, 1] - o[:, 0]).double().numpy()


SPLITS = {s: DATA[DATA["split"] == s].reset_index(drop=True) for s in ("train", "valid", "test")}
MANIFEST = {"backbone": BACKBONE_NAME, "split_file": SPLIT_FILE, "vocab_file": VOCAB_FILE, "concepts": CONCEPTS,
            "vocabularies": VOCABS, "concept_names": CONCEPT_NAMES, "head_specs": TCFG.HEAD_SPECS,
            "training": {k: getattr(TCFG, k) for k in dir(TCFG) if k.isupper() and k not in ("DEVICE",)},
            "seeds": {}}
MANIFEST["training"] = {k: (v if isinstance(v, (int, float, str, bool, list, tuple)) else str(v))
                        for k, v in MANIFEST["training"].items()}
metric_rows = []
if RUN_TRAINING:
    for seed in SEEDS:
        t0 = time.time()
        ck_path = train_concept_model(seed, SPLITS)
        ck = torch.load(ck_path, map_location=TCFG.DEVICE, weights_only=False)
        model = ConceptModel(BACKBONE_NAME, CONCEPTS, pretrained=False).to(TCFG.DEVICE)
        model.load_state_dict(ck["state_dict"])
        probs, hard = {}, {}
        for s in SPLITS:
            probs[s], hard[s] = concept_outputs(model, SPLITS[s], CONCEPTS)
            f1c, baccc, per = concept_metrics(hard[s], SPLITS[s], CONCEPTS)
            metric_rows.append({"seed": seed, "model": "concepts", "split": s, "concept_macro_f1": f1c,
                                "concept_balanced_accuracy": baccc, "best_epoch": ck["best_epoch"]})
        del model
        y = {s: SPLITS[s]["malignant"].values for s in SPLITS}
        heads, preds = {}, []
        for vname, kind in TCFG.HEAD_SPECS:
            cols = [CONCEPTS.index(c) for c in VOCABS[vname]]
            head, ep, vl = train_head(kind, cols, hard, y, seed)
            key = f"{vname}_{kind}"
            heads[key] = {"spec": head_spec(head), "state_dict": head.state_dict(), "kind": kind, "vocab": vname,
                          "columns": cols, "best_epoch": ep, "valid_loss": vl}
            for s in SPLITS:
                sc = head_logit_torch(head, hard[s][:, cols])
                metric_rows.append({"seed": seed, "model": key, "split": s, **label_metrics(y[s], sc), "best_epoch": ep})
                preds.append(pd.DataFrame({"seed": seed, "head": key, "split": s, "image_id": SPLITS[s]["image_id"],
                                           "y_true": y[s], "logit": sc, "y_pred": (sc > 0).astype(int)}))
        feat_path = os.path.join(MODELS_PATH, f"states_{BACKBONE_NAME}_seed{seed}.npz")
        np.savez_compressed(feat_path, **{f"prob_{s}": probs[s] for s in SPLITS}, **{f"hard_{s}": hard[s] for s in SPLITS},
                            **{f"y_{s}": y[s] for s in SPLITS},
                            **{f"id_{s}": SPLITS[s]["image_id"].values.astype(str) for s in SPLITS},
                            **{f"group_{s}": SPLITS[s]["group"].values for s in SPLITS})
        head_path = os.path.join(MODELS_PATH, f"heads_{BACKBONE_NAME}_seed{seed}.pth")
        torch.save(heads, head_path)
        pred_path = os.path.join(MODELS_PATH, f"predictions_{BACKBONE_NAME}_seed{seed}.csv")
        pd.concat(preds, ignore_index=True).to_csv(pred_path, index=False)
        MANIFEST["seeds"][str(seed)] = {"concept_checkpoint": ck_path, "states": feat_path, "heads": head_path,
                                        "predictions": pred_path, "seconds": time.time() - t0}
        original_print(f"seed {seed} finished in {time.time() - t0:.0f} s")
    with open(os.path.join(MODELS_PATH, "manifest_skincon.json"), "w") as fh:
        json.dump(MANIFEST, fh, indent=1)
    METRICS = pd.DataFrame(metric_rows)
    METRICS.to_csv(os.path.join(MODELS_PATH, "training_metrics.csv"), index=False)
    original_print("\nConcept predictors")
    original_print(METRICS[METRICS.model == "concepts"].pivot_table(index="seed", columns="split",
                   values="concept_macro_f1").round(4).to_string())
    original_print("\nDiagnosis heads on the test split (mean and sd over seeds)")
    t = METRICS[(METRICS.model != "concepts") & (METRICS.split == "test")]
    original_print(t.groupby("model")[["balanced_accuracy", "macro_f1", "sensitivity", "specificity", "auroc"]]
                   .agg(["mean", "std"]).round(4).to_string())

# Part 2. Algebraic interaction analysis on SkinCon

Part 2 trains nothing and never opens an image. It reads the predicted concept states and the diagnosis heads written by Part 1. Heads with at most 12 concepts are evaluated once on all 3^12 = 531,441 masked inputs, so every hard removal game is a lookup, the exact expected-hard game is a tensor contraction and every per-image quantity has a certified range over all 4,096 concept profiles. Heads with 16 and 20 concepts are evaluated directly on the 2^n masked inputs of each image.

The populations are the correctly classified malignant test images and the correctly classified non-malignant test images of each head, with every game oriented by the predicted class. The inverse temperature is frozen on validation images with the split-half rule, and the seed studied in depth has the highest validation balanced accuracy of the primary head. Everything is printed to the console, mirrored to `console_log.txt`, and the csv files are written in the last cell.

In [ ]:
# ============================================================
# PART 2 CONFIGURATION, CONSOLE LOG AND TABLE REGISTRY
# Part 2 reads the manifest of Part 1 and never touches the images. Every analysis choice that is not fixed in
# advance (the inverse temperature, the seed studied in depth) is frozen on validation images and stored in
# FROZEN_FILE, which later runs reuse.
# ============================================================
import os
import sys
import json
import time
import math
import pickle
import hashlib
import builtins
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format",
              lambda v: f"{v:.4f}" if (v == 0 or not np.isfinite(v) or 1e-3 <= abs(v) < 1e7) else f"{v:.3e}")


class Config:
    MODELS_DIR = MODELS_PATH
    OUTPUT_DIR = AIA_OUTPUT_PATH
    SEEDS = list(SEEDS)

    # ---------------- games ----------------
    PRIMARY_HEAD = "V12_mlp"                  # primary head, removal rule and game
    CONTROL_HEADS = ["V12_affine", "V12_mlp", "V12_deep"]
    SCALE_HEADS = ["V12_mlp", "V16_mlp", "V20_mlp"]
    REMOVALS = ["zero", "marginal"]           # hard games under both removal rules
    POPULATIONS = {"malignant": 1, "non_malignant": 0}
    POPULATION_LABEL = {"malignant": "malignant", "non_malignant": "non-malignant"}
    MIN_POPULATION_IMAGES = 10
    NUMERICAL_FLOOR = 1e-9                    # added to every measured table error

    # ---------------- ranking laws ----------------
    BETA0_GRID = [0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
    NULL_MC_DRAWS = 10000                     # Monte Carlo null scale above 12 concepts
    LARGE_N_IMAGES = 40                       # images per population and split on V16 and V20 (fixed random draw),
                                              # since each of their games needs up to 2^20 head evaluations

    # ---------------- certificates ----------------
    DELTA = 0.05                              # simultaneous level per seed, head, game and population
    N_BOOT = 2000

    # ---------------- experiment 1, computation ----------------
    MCMC_IMAGES = 4                           # test images per population and vocabulary, at quantiles of beta*osc
    MCMC_CHAINS = 8
    MCMC_STEPS = 100_000
    MCMC_RECORD_EVERY = 10
    MCMC_CHECKPOINTS = [1_000, 3_000, 10_000, 30_000, 100_000]
    RE_LADDER = [1.0, 0.5, 0.25, 0.125]
    RE_SWAP_EVERY = 10
    MIX_THETA = 0.01
    EH_IMAGES = 12                            # images per population for the Monte Carlo expected-hard game
    EH_DRAWS = [1_000, 10_000, 100_000]
    IDENTITY_IMAGES = 50                      # images per head with direct evaluation of every removal set

    # ---------------- experiment 3, summaries ----------------
    MAXENT_TOL = 1e-9
    MAXENT_MAX_IMAGES = 400                   # per population (all correctly classified test images when fewer)
    MAXENT_LARGE_IMAGES = 1                   # per population on V16 and V20 (median of beta*osc)
    MAXENT_TOL_LARGE = 1e-7                   # gradient tolerance on V16 and V20
    MAXENT_MAXITER_LARGE = 100                # iteration cap on V16 and V20 (the residual is reported)

    DEVICE = torch.device("cpu")


CFG = Config()
with open(os.path.join(CFG.MODELS_DIR, "manifest_skincon.json")) as _fh:
    MANIFEST = json.load(_fh)
_missing = [s for s in CFG.SEEDS if str(s) not in MANIFEST["seeds"]]
if _missing:
    raise ValueError(f"Seeds {_missing} have no trained models in the manifest. Run Part 1 for them first.")
CONCEPTS = MANIFEST["concepts"]
VOCABS = MANIFEST["vocabularies"]
CONCEPT_NAMES = MANIFEST["concept_names"]
HEAD_KEYS = [f"{v}_{k}" for v, k in MANIFEST["head_specs"]]
CFG.CACHE_DIR = os.path.join(CFG.OUTPUT_DIR, "cache")
CFG.CSV_DIR = os.path.join(CFG.OUTPUT_DIR, "csv")
for _d in (CFG.OUTPUT_DIR, CFG.CACHE_DIR, CFG.CSV_DIR):
    os.makedirs(_d, exist_ok=True)
CFG.FROZEN_FILE = os.path.join(CFG.OUTPUT_DIR, "frozen_choices.json")


def short(c):
    """Display name of a concept key."""
    nm = CONCEPT_NAMES.get(c, c)
    return nm.replace("(Hyperpigmentation)", "").replace("(Hypopigmentation)", "").strip()


def model_stamp(seed):
    """Fingerprint of the files of one seed (paths, sizes and modification times)."""
    e = MANIFEST["seeds"][str(seed)]
    parts = []
    for k in ("states", "heads"):
        st = os.stat(e[k])
        parts.append(f"{e[k]}|{st.st_size}|{int(st.st_mtime)}")
    return hashlib.md5("#".join(parts).encode()).hexdigest()[:10]


_LOG_PATH = os.path.join(CFG.OUTPUT_DIR, "console_log.txt")
with open(_LOG_PATH, "w", encoding="utf-8") as _fh:
    _fh.write(f"Console log started {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
TABLES = {}


def say(*args):
    msg = " ".join(str(a) for a in args)
    builtins.print(msg)
    sys.stdout.flush()
    with open(_LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(msg + "\n")


def banner(title, char="="):
    say("")
    say(char * 100)
    say(title)
    say(char * 100)


def show(df, title, key=None, echo=True):
    """Print a table in full and register it for the csv files written at the end."""
    if key is not None:
        TABLES.setdefault(key, []).append(df.assign(table=title) if "table" not in df.columns else df)
    if echo:
        say(f"\n--- {title} ---")
        say(df.to_string(index=False) if len(df) else "(empty)")


def _pkl_save(obj, path):
    with open(path, "wb") as fh:
        pickle.dump(obj, fh, protocol=4)


def _pkl_load(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)


def cached(name, fn):
    path = os.path.join(CFG.CACHE_DIR, f"{name}.pkl")
    if os.path.exists(path):
        return _pkl_load(path)
    out = fn()
    _pkl_save(out, path)
    return out


def frozen_choices():
    if os.path.exists(CFG.FROZEN_FILE):
        with open(CFG.FROZEN_FILE) as fh:
            return json.load(fh)
    return {}


def save_frozen(**kw):
    d = frozen_choices()
    for k, v in kw.items():
        d.setdefault(k, v)
    with open(CFG.FROZEN_FILE, "w") as fh:
        json.dump(d, fh, indent=1)
    return d


say(f"Seeds {CFG.SEEDS}, backbone {MANIFEST['backbone']}, heads {HEAD_KEYS}")
say("Vocabularies: " + "; ".join(f"{k} = {', '.join(short(c) for c in v)}" for k, v in VOCABS.items()))

## Library

In [ ]:
# ============================================================
# RANKING-LAW AND INTERACTION LIBRARY
# Subsets of C = {0,...,n-1} are bitmasks (bit j <=> concept j). Removal games u(A) record the score after the
# concepts in A are hidden, presence games v(K) = u(C minus K).
#   ranking laws    SOFI score in Weyl coordinates (Lemma 1), forward-backward recursions (Proposition 2 i),
#                   max-sum recursion for the SOFI optimum, exact null moments of the score
#   games           Walsh analysis with the odd/even split (Proposition 4), pairwise indices of every method,
#                   parity duality of symmetric indices (Proposition 6), decision diagnostics
#   computation     edge-weighted recursions for position-feature laws (exact maximum-entropy fits without
#                   enumerating orderings), exact i.i.d. sampling, lazy Metropolis with replica exchange,
#                   rank-normalized split R-hat and bulk effective sample size
#   certificates    Hoeffding and empirical Bernstein radii (Maurer and Pontil 2009), images needed
# Nothing here enumerates the n! orderings, so every function runs for n up to 20.
# ============================================================
import itertools
import math
import numpy as np
from scipy.special import logsumexp, comb
from scipy.optimize import minimize, brentq
from scipy.stats import norm as _norm
from scipy import stats as sps


LOG_ZERO = -np.inf


class Lattice:
    """Boolean lattice of subsets of C with the index tables used by every recursion."""

    def __init__(self, n):
        self.n = int(n)
        self.N = 1 << self.n
        idx = np.arange(self.N)
        self.idx = idx
        self.mem = ((idx[:, None] >> np.arange(self.n)) & 1).astype(bool)      # (N, n)
        self.size = self.mem.sum(1)
        self.full = self.N - 1
        self.comp = self.full ^ idx
        bits = (1 << np.arange(self.n))
        self.rm = np.where(self.mem, idx[:, None] ^ bits[None, :], -1)          # S \ {j}
        self.add = np.where(~self.mem, idx[:, None] | bits[None, :], -1)        # S U {j}
        self.layers = [np.where(self.size == k)[0] for k in range(self.n + 1)]
        self.interior = (self.size > 0) & (self.size < self.n)


_LATTICES = {}


def lattice(n):
    if n not in _LATTICES:
        _LATTICES[n] = Lattice(n)
    return _LATTICES[n]


def weyl_rho(n):
    """Weyl vector of sl_n, rho_k = (n+1)/2 - k for positions k = 1..n."""
    return (n + 1) / 2.0 - np.arange(1, n + 1, dtype=float)


def rho_norm2(n):
    return n * (n * n - 1) / 12.0


# ------------------------------------------------------------
# Removal games and the ranking score
# ------------------------------------------------------------


def pair_list(n):
    return [(a, b) for a in range(n) for b in range(a + 1, n)]


def eta_from_u(u, lat):
    """eta_x(S) = u_x(C minus S) - u_x(S); u has shape (L, 2^n)."""
    u = np.atleast_2d(u)
    return u[:, lat.comp] - u


def deg_from_eta_perm(eta, perms, lat):
    """Score of each ordering as deg(sigma) = sum_k eta(S_k), summed over the prefixes of sigma."""
    n = lat.n
    bits = (1 << perms)                            # (P, n)
    pref = np.cumsum(bits, axis=1)[:, :n - 1]      # prefixes S_1..S_{n-1}
    eta = np.atleast_2d(eta)
    return eta[:, pref].sum(-1)                    # (L, P)


def gibbs_sofi_dp(u, beta, lat=None, want_pairs=True, want_chain=True, chunk=None):
    """
    Exact Gibbs-SOFI quantities for a batch of games u (L, 2^n) at inverse temperature beta
    (scalar or (L,) array). Returns dict with logZ, P (L,n,n) [attribute, position],
    pair (L,n,n) with pair[a,b] = mu(a before b), chain (L,2^n) chain-passage probabilities,
    Edeg, entropy H. Log-sum-exp recursions (numerically stable).
    """
    u = np.atleast_2d(np.asarray(u, dtype=float))
    L = u.shape[0]
    n = int(round(math.log2(u.shape[1])))
    lat = lat or lattice(n)
    beta = np.broadcast_to(np.asarray(beta, dtype=float), (L,)).copy()
    if chunk is not None and L > chunk:
        parts = [gibbs_sofi_dp(u[i:i + chunk], beta[i:i + chunk], lat, want_pairs, want_chain)
                 for i in range(0, L, chunk)]
        return {k: np.concatenate([p[k] for p in parts], 0) for k in parts[0]}
    eta = eta_from_u(u, lat)
    logw = beta[:, None] * eta
    logw[:, 0] = 0.0
    logw[:, lat.full] = 0.0
    logF = np.full((L, lat.N), LOG_ZERO)
    logF[:, 0] = 0.0
    for k in range(1, n + 1):
        S = lat.layers[k]
        prev = lat.rm[S]                                     # (m, n), -1 where j not in S
        vals = np.where(prev[None] >= 0, logF[:, np.maximum(prev, 0)], LOG_ZERO)
        logF[:, S] = logw[:, S] + logsumexp(vals, axis=-1)
    logB = np.full((L, lat.N), LOG_ZERO)
    logB[:, lat.full] = 0.0
    for k in range(n - 1, -1, -1):
        S = lat.layers[k]
        nxt = lat.add[S]
        vals = np.where(nxt[None] >= 0,
                        logw[:, np.maximum(nxt, 0)] + logB[:, np.maximum(nxt, 0)], LOG_ZERO)
        logB[:, S] = logsumexp(vals, axis=-1)
    logZ = logB[:, 0]
    out = {"logZ": logZ, "logZ_forward": logF[:, lat.full]}
    # q(S, j) = F(S) w(S+j) B(S+j) / Z  for j not in S
    addc = np.maximum(lat.add, 0)
    logq = np.where(lat.add[None] >= 0,
                    logF[:, :, None] + logw[:, addc] + logB[:, addc] - logZ[:, None, None],
                    LOG_ZERO)
    q = np.exp(logq)                                          # (L, N, n)
    P = np.zeros((L, n, n))
    for k in range(n):
        P[:, :, k] = q[:, lat.layers[k], :].sum(1)
    out["P"] = P
    if want_pairs:
        out["pair"] = np.einsum("sa,lsb->lab", lat.mem.astype(float), q)
    chain = np.exp(logF + logB - logZ[:, None])
    if want_chain:
        out["chain"] = chain
    inter = lat.interior
    Edeg = (chain[:, inter] * eta[:, inter]).sum(1)
    out["Edeg"] = Edeg
    out["H"] = logZ - beta * Edeg
    return out


def deg_extremes_dp(u, lat=None, tol_rel=1e-9):
    """
    Max-sum and min-sum recursions with counts of optimal chains (the beta to infinity limit).
    Returns dmax, dmin, one argmax ordering (lowest attribute index tie-break), the number of
    maximizing orderings, and the hard-limit law (uniform on all maximizers): Pinf, pairinf.
    """
    u = np.atleast_2d(np.asarray(u, dtype=float))
    L = u.shape[0]
    n = int(round(math.log2(u.shape[1])))
    lat = lat or lattice(n)
    eta = eta_from_u(u, lat)
    et = eta.copy()
    et[:, 0] = 0.0
    et[:, lat.full] = 0.0
    scale = 1.0 + np.abs(et).max(1)
    tol = tol_rel * scale
    res = {}
    for mode in ("max", "min"):
        sgn = 1.0 if mode == "max" else -1.0
        e = sgn * et
        G = np.full((L, lat.N), -np.inf)
        G[:, 0] = 0.0
        cF = np.zeros((L, lat.N))
        cF[:, 0] = 1.0
        for k in range(1, n + 1):
            S = lat.layers[k]
            prev = lat.rm[S]
            vals = np.where(prev[None] >= 0, G[:, np.maximum(prev, 0)], -np.inf)
            best = vals.max(-1)
            G[:, S] = e[:, S] + best
            hit = vals >= best[..., None] - tol[:, None, None]
            cnt = np.where(hit & (prev[None] >= 0), cF[:, np.maximum(prev, 0)], 0.0)
            cF[:, S] = cnt.sum(-1)
        res[mode] = sgn * G[:, lat.full]
        if mode == "max":
            GF, CF = G, cF
    # backward max with counts
    GB = np.full((L, lat.N), -np.inf)
    GB[:, lat.full] = 0.0
    cB = np.zeros((L, lat.N))
    cB[:, lat.full] = 1.0
    for k in range(n - 1, -1, -1):
        S = lat.layers[k]
        nxt = lat.add[S]
        vals = np.where(nxt[None] >= 0, et[:, np.maximum(nxt, 0)] + GB[:, np.maximum(nxt, 0)], -np.inf)
        best = vals.max(-1)
        GB[:, S] = best
        hit = vals >= best[..., None] - tol[:, None, None]
        cB[:, S] = np.where(hit & (nxt[None] >= 0), cB[:, np.maximum(nxt, 0)], 0.0).sum(-1)
    dmax = res["max"]
    addc = np.maximum(lat.add, 0)
    through = GF[:, :, None] + et[:, addc] + GB[:, addc]
    opt = (lat.add[None] >= 0) & (through >= dmax[:, None, None] - tol[:, None, None])
    ways = np.where(opt, CF[:, :, None] * cB[:, addc], 0.0)          # (L, N, n)
    total = CF[:, lat.full]
    Pinf = np.zeros((L, n, n))
    for k in range(n):
        Pinf[:, :, k] = ways[:, lat.layers[k], :].sum(1)
    Pinf /= total[:, None, None]
    pairinf = np.einsum("sa,lsb->lab", lat.mem.astype(float), ways) / total[:, None, None]
    # lexicographically smallest optimal ordering, built forward along optimal transitions
    argmax_perm = np.zeros((L, n), dtype=int)
    for l in range(L):
        S = 0
        for k in range(n):
            for j in range(n):
                if not (S >> j) & 1 and opt[l, S, j]:
                    argmax_perm[l, k] = j
                    S |= (1 << j)
                    break
    return {"dmax": dmax, "dmin": res["min"], "n_opt": total, "argmax_perm": argmax_perm,
            "Pinf": Pinf, "pairinf": pairinf}


# ------------------------------------------------------------
# Score scale and inverse temperature
# ------------------------------------------------------------


def null_moments(u, lat=None):
    """
    Exact moments of deg under a uniformly random ordering, which give the null reference level.
    m_x(j,k) = E0[deg 1[sigma(k)=j]] from subset sums, and E0[deg^2] from nested pairs (O(3^n)).
    """
    u = np.atleast_2d(np.asarray(u, float))
    L = u.shape[0]
    n = int(round(math.log2(u.shape[1])))
    lat = lat or lattice(n)
    eta = eta_from_u(u, lat)
    m = np.zeros((L, n, n))
    for l in range(1, n):
        S = lat.layers[l]
        w = 1.0 / comb(n, l)
        memS = lat.mem[S].astype(float)                       # (m, n)
        e = eta[:, S] * w                                     # (L, m)
        in_part = np.einsum("lm,mj->lj", e, memS) / l          # attribute in S, positions k <= l
        out_part = np.einsum("lm,mj->lj", e, 1 - memS) / (n - l)
        m[:, :, :l] += in_part[:, :, None]
        m[:, :, l:] += out_part[:, :, None]
    # second moment
    E2 = np.zeros(L)
    for k in range(1, n):
        S = lat.layers[k]
        E2 += (eta[:, S] ** 2).mean(1)
    # nested pairs S subset T, 1<=|S|<|T|<=n-1, uniform over nested pairs of given sizes
    for (s, t), arr in _nested_pairs(n).items():
        E2 += 2.0 * (eta[:, arr[:, 0]] * eta[:, arr[:, 1]]).mean(1)
    return m, E2


_NESTED = {}


def _nested_pairs(n):
    if n not in _NESTED:
        lat = lattice(n)
        acc = {}
        for T in range(lat.N):
            t = lat.size[T]
            if t < 2 or t > n - 1:
                continue
            sub = (T - 1) & T
            while sub > 0:
                acc.setdefault((int(lat.size[sub]), int(t)), []).append((sub, T))
                sub = (sub - 1) & T
        _NESTED[n] = {k: np.array(v) for k, v in acc.items()}
    return _NESTED[n]


_SHAPLEY_G = {}


def shapley_values(u, lat):
    """Exact Shapley values of the presence game val(K) = u(C minus K), vectorized over games."""
    n = lat.n
    if n not in _SHAPLEY_G:
        G = np.zeros((lat.N, n))
        fact = [math.factorial(i) for i in range(n + 1)]
        for S in range(lat.N):
            s = int(lat.size[S])
            for j in range(n):
                if not (S >> j) & 1:
                    w = fact[s] * fact[n - s - 1] / fact[n]
                    G[S | (1 << j), j] += w
                    G[S, j] -= w
        _SHAPLEY_G[n] = G
    val = np.atleast_2d(u)[:, lat.comp]
    return val @ _SHAPLEY_G[n]


def walsh(u):
    """Walsh coefficients hat u(T) = 2^-n sum_A u(A) (-1)^{|A cap T|}, fast transform along the last axis."""
    a = np.array(np.atleast_2d(u), dtype=float, copy=True)
    N = a.shape[1]
    h = 1
    while h < N:
        a = a.reshape(a.shape[0], -1, 2 * h)
        x, y = a[:, :, :h].copy(), a[:, :, h:].copy()
        a[:, :, :h], a[:, :, h:] = x + y, x - y
        a = a.reshape(a.shape[0], N)
        h *= 2
    return a / N


def inverse_walsh(c):
    """Inverse transform, u(A) = sum_T hat u(T) chi_T(A)."""
    return walsh(c) * np.atleast_2d(c).shape[1]


def walsh_diagnostics(u, lat):
    """
    Non-additivity index with its odd/even split (Propositions 4 and 5 vi), Banzhaf removal effects
    c_j = 2 hat u({j}) and Banzhaf interaction indices I(a,b) = 4 hat u({a,b}), per game.
    """
    n = lat.n
    co = walsh(u)
    sz = lat.size
    e2 = co ** 2
    V = e2[:, sz >= 1].sum(1)                                   # Var(u) over a uniform subset
    H = e2[:, sz >= 2].sum(1)                                   # non-additive energy
    Hodd = e2[:, (sz >= 3) & (sz % 2 == 1)].sum(1)
    Heven = e2[:, (sz >= 2) & (sz % 2 == 0)].sum(1)
    safe = np.where(V > 0, V, 1.0)
    NA = np.where(V > 0, H / safe, 0.0)
    NAo = np.where(V > 0, Hodd / safe, 0.0)
    NAe = np.where(V > 0, Heven / safe, 0.0)
    order_share = np.stack([np.where(V > 0, e2[:, sz == k].sum(1) / safe, 0.0) for k in range(1, n + 1)], 1)
    single = [1 << j for j in range(n)]
    pairs = [(1 << a) | (1 << b) for a, b in pair_list(n)]
    return {"coef": co, "V": V, "H": H, "Hodd": Hodd, "Heven": Heven, "NA": NA, "NA_odd": NAo, "NA_even": NAe,
            "order_share": order_share, "effects": 2 * co[:, single], "I": 4 * co[:, pairs]}


def additive_projection(u, lat):
    """Best additive approximation in uniform-subset L2 (Walsh coefficients of order at most one)."""
    co = walsh(u)
    co[:, lat.size >= 2] = 0.0
    return inverse_walsh(co)


def parity_parts(u, lat):
    """Split u = mean + odd part + even part, with the even part complement-even and mean-free."""
    u = np.atleast_2d(u)
    co = walsh(u)
    odd = co.copy(); odd[:, lat.size % 2 == 0] = 0.0
    even = co.copy(); even[:, (lat.size % 2 == 1) | (lat.size == 0)] = 0.0
    return co[:, 0], inverse_walsh(odd), inverse_walsh(even)


def certified_na(W, eps):
    """Certificate of non-additivity with the interval for NA under table error eps (Section 5.1, eq. 55-56)."""
    sH, sV = np.sqrt(W["H"]), np.sqrt(W["V"])
    cert = sH > eps
    lo = np.where(sV > eps, np.maximum(sH - eps, 0) ** 2 / (sV + eps) ** 2, 0.0)
    hi = np.where(sV > eps, np.minimum(1.0, (sH + eps) ** 2 / np.maximum(sV - eps, 1e-300) ** 2), 1.0)
    return cert, lo, hi


def banzhaf_interaction_brute(u, lat):
    """Mean second difference over backgrounds, the definition of Proposition 5 (vi), for the identity check."""
    n = lat.n
    out = np.zeros((np.atleast_2d(u).shape[0], len(pair_list(n))))
    for i, (a, b) in enumerate(pair_list(n)):
        ba, bb = 1 << a, 1 << b
        A = np.where(~lat.mem[:, a] & ~lat.mem[:, b])[0]
        d = u[:, A | ba | bb] - u[:, A | ba] - u[:, A | bb] + u[:, A]
        out[:, i] = d.mean(1)
    return out


def _second_difference_operator(n, weight_of_k):
    """Matrix Q (m2 x 2^n) with (Q v)_{ab} = sum_{K subset N minus ab} w(|K|) [v(K+ab) - v(K+a) - v(K+b) + v(K)]."""
    lat = lattice(n)
    Q = np.zeros((len(pair_list(n)), lat.N))
    for i, (a, b) in enumerate(pair_list(n)):
        ba, bb = 1 << a, 1 << b
        for K in np.where(~lat.mem[:, a] & ~lat.mem[:, b])[0]:
            w = weight_of_k(int(lat.size[K]))
            Q[i, K | ba | bb] += w; Q[i, K | ba] -= w; Q[i, K | bb] -= w; Q[i, K] += w
    return Q


_IDX_CACHE = {}


def index_operators(n):
    """Linear operators on presence games v(K) = u(C minus K) for the pairwise indices of every method."""
    if n in _IDX_CACHE:
        return _IDX_CACHE[n]
    lat = lattice(n)
    f = math.factorial
    ops = {
        "banzhaf": _second_difference_operator(n, lambda k: 1.0 / 2 ** (n - 2)),
        "shapley_interaction": _second_difference_operator(n, lambda k: f(k) * f(n - k - 2) / f(n - 1)),
        "shapley_taylor": _second_difference_operator(n, lambda k: (2.0 / n) / math.comb(n - 1, k)),
    }
    # Faith indices of order two: weighted least squares over the Moebius basis of degree at most two
    basis = [0] + [1 << j for j in range(n)] + [(1 << a) | (1 << b) for a, b in pair_list(n)]
    X = np.array([[1.0 if (T & S) == T else 0.0 for T in basis] for S in range(lat.N)])   # T subset S
    size = lat.size
    kern = np.zeros(lat.N)
    inner = (size > 0) & (size < n)
    kern[inner] = (n - 1) / (np.array([math.comb(n, int(s)) for s in size[inner]]) * size[inner] * (n - size[inner]))
    # Faith-Shap, Shapley kernel with exact fit at the empty and the full coalition (KKT system)
    Wd = np.diag(kern)
    Cc = X[[0, lat.full]]
    Kkt = np.block([[X.T @ Wd @ X, Cc.T], [Cc, np.zeros((2, 2))]])
    rhs = np.vstack([X.T @ Wd, np.eye(lat.N)[[0, lat.full]]])
    L_fs = np.linalg.solve(Kkt, rhs)[:len(basis)]
    # Faith-Banzhaf, uniform weights without constraints
    L_fb = np.linalg.solve(X.T @ X, X.T)
    ops["faith_shap"] = L_fs[1 + n:]
    ops["faith_banzhaf"] = L_fb[1 + n:]
    out = {"ops": ops, "faith_full": {"faith_shap": L_fs, "faith_banzhaf": L_fb}, "moebius_X": X, "kernel": kern}
    _IDX_CACHE[n] = out
    return out


def interaction_indices(u, lat, W=None):
    """Every pairwise index per game (rows) and pair (columns), on the removal game u."""
    n = lat.n
    v = np.atleast_2d(u)[:, lat.comp]                       # presence game
    OP = index_operators(n)
    out = {k: v @ Q.T for k, Q in OP["ops"].items()}
    W = W or walsh_diagnostics(u, lat)
    co = W["coef"]
    Hs = []
    for a, b in pair_list(n):
        num = co[:, (1 << a) | (1 << b)] ** 2
        den = co[:, 1 << a] ** 2 + co[:, 1 << b] ** 2 + num
        Hs.append(np.where(den > 0, num / np.where(den > 0, den, 1.0), 0.0))
    out["h_statistic"] = np.stack(Hs, 1)
    return out


def mixing_time_bound_log10(beta, osc, n, theta):
    """log10 of the bound of Proposition 2(ii), computed in logarithms to avoid overflow."""
    lg = -2 * beta * osc + math.log((1 - math.cos(math.pi / n)) / (n - 1))
    num = math.lgamma(n + 1) + beta * osc + math.log(1 / theta)
    return (math.log(num) - lg) / math.log(10)


# ------------------------------------------------------------
# Rank-normalized split R-hat and bulk effective sample size (Vehtari et al. 2021)
# ------------------------------------------------------------


def _split(x):
    """x (draws, chains) -> split chains (draws // 2, 2 * chains)."""
    h = x.shape[0] // 2
    return np.concatenate([x[:h], x[h:2 * h]], axis=1)


def _rank_normal(x):
    r = sps.rankdata(x, method="average").reshape(x.shape)
    return _norm.ppf((r - 3 / 8) / (x.size + 1 / 4))


def _rhat_basic(x):
    n, m = x.shape
    if n < 2 or m < 2:
        return np.nan
    w = x.var(0, ddof=1).mean()
    b = n * x.mean(0).var(ddof=1)
    if w <= 0:
        return np.nan
    return float(np.sqrt(((n - 1) / n * w + b / n) / w))


def rhat_rank(x):
    """Maximum of the bulk and tail rank-normalized split R-hat of draws x (draws, chains)."""
    s = _split(np.asarray(x, float))
    bulk = _rhat_basic(_rank_normal(s))
    tail = _rhat_basic(_rank_normal(np.abs(s - np.median(s))))
    return float(np.nanmax([bulk, tail]))


def ess_bulk(x):
    """Bulk effective sample size with Geyer's initial monotone sequence on rank-normalized split chains."""
    z = _rank_normal(_split(np.asarray(x, float)))
    n, m = z.shape
    if n < 4:
        return np.nan
    zc = z - z.mean(0)
    f = np.fft.rfft(zc, n=2 * n, axis=0)
    acov = np.fft.irfft(f * np.conj(f), axis=0)[:n] / n
    w = acov[0] * n / (n - 1)
    var_plus = w.mean() * (n - 1) / n + z.mean(0).var(ddof=1) if m > 1 else w.mean()
    if var_plus <= 0:
        return np.nan
    rho = 1 - (w.mean() - acov.mean(1)) / var_plus
    rho[0] = 1.0
    s, t = 0.0, 0
    prev = np.inf
    while t + 1 < n:
        p = rho[t] + rho[t + 1]
        if p < 0:
            break
        p = min(p, prev)
        s += p
        prev = p
        t += 2
    tau = -1 + 2 * s
    return float(n * m / max(tau, 1e-12))


# ------------------------------------------------------------
# k-medoids on a distance matrix (adapter of global attribution mapping)
# ------------------------------------------------------------


def holm(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    adj = np.empty(m)
    run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * pv[i]))
        adj[i] = run
    out[ok] = adj
    return out


def boot_ci(x, rng, B=None, stat=None):
    """Case-bootstrap percentile interval of a mean (vectorized over resamples)."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    B = B or CFG.N_BOOT
    bs = x[rng.integers(0, len(x), (B, len(x)))].mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def _lse(v, axis):
    """Log-sum-exp along one axis (faster than the general scipy routine for these arrays)."""
    m = v.max(axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    with np.errstate(divide="ignore"):
        return np.log(np.exp(v - m).sum(axis=axis)) + np.squeeze(m, axis)


def edge_dp(logE, lat, want_pairs=True):
    """
    Law p(sigma) proportional to prod_k E(S_{k-1}, sigma(k)) over the edges of the maximal chain of sigma.
    logE has shape (N, n) with entries for j not in S (others ignored). Returns logZ, P (n, n) [attribute,
    position], pair (n, n) with pair[a, b] = p(a before b), the entropy H and q (N, n) edge-passage probabilities.
    """
    n, N = lat.n, lat.N
    notin = lat.add >= 0                                             # (N, n): j not in S
    logF = np.full(N, -np.inf)
    logF[0] = 0.0
    for k in range(1, n + 1):
        S = lat.layers[k]
        prev = lat.rm[S]                                             # S minus j, -1 when j not in S
        pc = np.maximum(prev, 0)
        jj = np.broadcast_to(np.arange(n), prev.shape)
        vals = np.where(prev >= 0, logF[pc] + logE[pc, jj], -np.inf)
        logF[S] = _lse(vals, 1)
    logB = np.full(N, -np.inf)
    logB[lat.full] = 0.0
    for k in range(n - 1, -1, -1):
        S = lat.layers[k]
        nxt = lat.add[S]
        nc = np.maximum(nxt, 0)
        vals = np.where(nxt >= 0, logE[S] + logB[nc], -np.inf)
        logB[S] = _lse(vals, 1)
    logZ = logB[0]
    addc = np.maximum(lat.add, 0)
    logq = np.where(notin, logF[:, None] + logE + logB[addc] - logZ, -np.inf)
    q = np.exp(logq)
    P = np.zeros((n, n))
    for k in range(n):
        P[:, k] = q[lat.layers[k]].sum(0)
    out = {"logZ": float(logZ), "P": P, "q": q,
           "H": float(logZ - np.where(notin, q * np.where(notin, logE, 0.0), 0.0).sum())}
    if want_pairs:
        out["pair"] = lat.mem.astype(float).T @ q                    # pair[a, b] = sum over S containing a of q(S, b)
    return out


def position_logE(theta, lat):
    """Edge log-weights theta[j, |S|] of a position-feature law."""
    return theta.T[np.minimum(lat.size, lat.n - 1)]                             # (N, n): row S -> theta[:, |S|]


def maxent_positions(target, lat, kind="P", tol=1e-10, maxiter=5000):
    """
    Maximum-entropy law with prescribed soft ranking (kind 'P', target n x n) or prescribed average positions
    (kind 'y', target of length n in Weyl coordinates). Returns theta, logZ, the residual of the moment match and
    the function kl(P_mu, H_mu) that gives the exact KL divergence from a law mu to the fitted law.
    """
    n = lat.n
    rho = (n + 1) / 2.0 - np.arange(1, n + 1)

    def unpack(x):
        return x.reshape(n, n) if kind == "P" else np.outer(x, rho)

    def fg(x):
        th = unpack(x)
        d = edge_dp(position_logE(th, lat), lat, want_pairs=False)
        if kind == "P":
            return d["logZ"] - (th * target).sum(), (d["P"] - target).ravel()
        y = d["P"] @ rho
        return d["logZ"] - x @ target, y - target

    x0 = np.zeros(n * n if kind == "P" else n)
    res = minimize(fg, x0, jac=True, method="L-BFGS-B", options={"maxiter": maxiter, "gtol": tol, "ftol": 1e-15})
    th = unpack(res.x)
    d = edge_dp(position_logE(th, lat), lat, want_pairs=True)
    resid = float(np.abs((d["P"] - target) if kind == "P" else (d["P"] @ rho - target)).max())

    def kl(P_mu, H_mu):
        return float(-H_mu - (th * P_mu).sum() + d["logZ"])
    return {"theta": th, "logZ": d["logZ"], "P": d["P"], "pair": d["pair"], "H": d["H"], "resid": resid,
            "iterations": int(res.nit), "kl": kl}


def gibbs_backward(u, beta, lat):
    """Node log-weights and backward sums of the Gibbs-SOFI law of one game (for exact sampling)."""
    eta = u[lat.comp] - u
    logw = beta * eta
    logw[0] = 0.0
    logw[lat.full] = 0.0
    logB = np.full(lat.N, -np.inf)
    logB[lat.full] = 0.0
    for k in range(lat.n - 1, -1, -1):
        S = lat.layers[k]
        nxt = lat.add[S]
        nc = np.maximum(nxt, 0)
        logB[S] = logsumexp(np.where(nxt >= 0, logw[nc] + logB[nc], -np.inf), axis=1)
    return logw, logB


def exact_samples(u, beta, lat, m, rng):
    """m independent orderings from the Gibbs-SOFI law, drawn step by step from the backward weights."""
    logw, logB = gibbs_backward(u, beta, lat)
    n = lat.n
    S = np.zeros(m, dtype=np.int64)
    perms = np.zeros((m, n), dtype=np.int64)
    for k in range(n):
        nxt = lat.add[S]                                             # (m, n)
        nc = np.maximum(nxt, 0)
        lp = np.where(nxt >= 0, logw[nc] + logB[nc] - logB[S][:, None], -np.inf)
        p = np.exp(lp - logsumexp(lp, axis=1, keepdims=True))
        c = (p.cumsum(1) > rng.random(m)[:, None]).argmax(1)
        perms[:, k] = c
        S = S | (1 << c)
    return perms


def position_counts(perms, n):
    pos = np.argsort(perms, axis=1)
    Pc = np.zeros((n, n))
    np.add.at(Pc, (perms, np.broadcast_to(np.arange(n), perms.shape)), 1.0)
    pr = (pos[:, :, None] < pos[:, None, :]).sum(0).astype(float)
    return Pc, pr


def run_chains(eta, beta, img_of_chain, steps, record_every, snap_steps, rng, ladder=(1.0,), swap_every=10, pairs=True):
    """
    Lazy Metropolis chains over adjacent transpositions with optional replica exchange. Only the replica at the
    target temperature (first ladder entry) is recorded. Position counts (and pair counts) are accumulated at
    every recorded step, with cumulative snapshots at the steps in snap_steps, so any window [t0, t1] of two
    snapshots gives an empirical soft ranking. Returns snapshots {step: (Pc, Pr)} per chain, the recorded scores
    (records, chains) and the swap acceptance rate.
    """
    n = eta.shape[1].bit_length() - 1
    C = len(img_of_chain)
    K = len(ladder)
    img = np.repeat(np.asarray(img_of_chain), K)
    bet = (np.asarray(beta, float)[np.asarray(img_of_chain)][:, None] * np.asarray(ladder)[None, :]).ravel()
    R = C * K
    perm = np.stack([rng.permutation(n) for _ in range(R)]).astype(np.int64)
    pref = np.concatenate([np.zeros((R, 1), np.int64), np.cumsum(1 << perm, axis=1)], axis=1)
    Ns = eta.shape[1]
    flat = eta.ravel()
    deg = flat[img[:, None] * Ns + pref[:, 1:n]].sum(1)
    rows = np.arange(R)
    target = np.arange(C) * K
    Pc = np.zeros((C, n, n))
    Pr = np.zeros((C, n, n)) if pairs else None
    snaps = {}
    nrec = steps // record_every
    rec_deg = np.zeros((nrec, C))
    sw_try = sw_ok = 0
    cidx = np.arange(C)[:, None]
    kpos = np.arange(n)[None, :]
    snap_set = set(snap_steps)
    for t in range(1, steps + 1):
        move = rng.random(R) >= 0.5
        e = rng.integers(0, n - 1, R)
        a = perm[rows, e]
        b = perm[rows, e + 1]
        old = pref[rows, e + 1]
        new = old ^ (1 << a) ^ (1 << b)
        dlt = flat[img * Ns + new] - flat[img * Ns + old]
        acc = move & (np.log(rng.random(R)) < bet * dlt)
        if acc.any():
            r = rows[acc]
            ee = e[acc]
            perm[r, ee], perm[r, ee + 1] = b[acc], a[acc]
            pref[r, ee + 1] = new[acc]
            deg[r] += dlt[acc]
        if K > 1 and t % swap_every == 0:
            start = (t // swap_every) % 2
            for k in range(start, K - 1, 2):
                i1 = np.arange(C) * K + k
                i2 = i1 + 1
                ok = np.log(rng.random(C)) < (bet[i1] - bet[i2]) * (deg[i2] - deg[i1])
                sw_try += C
                sw_ok += int(ok.sum())
                if ok.any():
                    s1, s2 = i1[ok], i2[ok]
                    for arr in (perm, pref, deg):
                        tmp = arr[s1].copy(); arr[s1] = arr[s2]; arr[s2] = tmp
        if t % record_every == 0:
            pt = perm[target]
            Pc[cidx, pt, kpos] += 1.0
            if pairs:
                pos = np.argsort(pt, axis=1)
                Pr += pos[:, :, None] < pos[:, None, :]
            rec_deg[t // record_every - 1] = deg[target]
        if t in snap_set:
            snaps[t] = (Pc.copy(), None if Pr is None else Pr.copy())
    return snaps, rec_deg, (sw_ok / sw_try if sw_try else np.nan)


def null_sd_mc(u, lat, rng, m=10000):
    """Monte Carlo standard deviation of deg under uniform orderings, per game (rows of u)."""
    u = np.atleast_2d(u)
    perms = np.stack([rng.permutation(lat.n) for _ in range(m)])
    eta = u[:, lat.comp] - u
    bits = (1 << perms)
    pref = np.cumsum(bits, axis=1)[:, :lat.n - 1]
    out = np.zeros(u.shape[0])
    for i in range(u.shape[0]):
        out[i] = eta[i][pref].sum(1).std()
    return out


def empirical_bernstein(x, length, delta):
    """Two-sided radius of the empirical Bernstein bound (Maurer and Pontil 2009) at level 1 - delta."""
    x = np.asarray(x, float)
    N = len(x)
    if N < 2:
        return np.inf
    v = x.var(ddof=1)
    lg = math.log(4.0 / delta)
    return math.sqrt(2 * v * lg / N) + 7 * length * lg / (3 * (N - 1))


def hoeffding(length, N, delta):
    return length * math.sqrt(math.log(2.0 / delta) / (2 * N))


def symmetric_difference_operator(n, k, weight):
    """
    Matrix of the order-k difference index sum_{A subset C minus K} w(|A|) Delta_K u(A) on removal games, for every
    K of size k (rows ordered by itertools.combinations). Proposition 6 covers weights with w(t) = w(n - k - t).
    """
    import itertools
    lat_N = 1 << n
    Ks = list(itertools.combinations(range(n), k))
    Q = np.zeros((len(Ks), lat_N))
    for i, K in enumerate(Ks):
        kmask = sum(1 << j for j in K)
        others = [j for j in range(n) if j not in K]
        for r in range(len(others) + 1):
            for A in itertools.combinations(others, r):
                amask = sum(1 << j for j in A)
                w = weight(r)
                for L in range(kmask + 1):
                    if L & ~kmask:
                        continue
                    sign = (-1) ** (k - bin(L).count("1"))
                    Q[i, amask | L] += w * sign
    return Ks, Q


# ------------------------------------------------------------
# Additions for larger vocabularies, decisions and certificates
# ------------------------------------------------------------

def gibbs_logE(u, beta, lat):
    """Edge log-weights of the Gibbs-SOFI law of one game, logE(S, j) = beta * eta(S + j) (zero at the full set)."""
    eta = u[lat.comp] - u
    logw = beta * eta
    logw[0] = 0.0
    logw[lat.full] = 0.0
    return logw[np.maximum(lat.add, 0)]


def gibbs_single(u, beta, lat, want_pairs=True):
    """Exact Gibbs-SOFI law of one game through the edge recursions (memory O(n 2^n), used for n > 12)."""
    return edge_dp(gibbs_logE(u, beta, lat), lat, want_pairs=want_pairs)


def null_sd(u, lat, rng, mc_draws=10000, exact_max_n=12):
    """Null standard deviation of the score per game, exact from nested pairs up to exact_max_n, else Monte Carlo."""
    u = np.atleast_2d(u)
    if lat.n <= exact_max_n:
        out = []
        for i in range(0, len(u), 64):
            _, E2 = null_moments(u[i:i + 64], lat)
            out.append(np.sqrt(np.maximum(E2, 0.0)))
        return np.concatenate(out) if out else np.zeros(0), "exact"
    return null_sd_mc(u, lat, rng, mc_draws), f"monte_carlo_{mc_draws}"


def decision_diagnostics(u, lat):
    """
    Decision-level effects of non-additivity for games oriented so that u(empty set) > 0 is the predicted class.
    joint_flip          some pair a, b keeps the decision when a or b is hidden alone but flips it when both are hidden
    interaction_flip    some pair a, b whose joint removal flips the decision against the additive prediction
                        u(a) + u(b) - u(empty set), so the flip needs the second difference
    subset shares       fraction of the 2^n removal sets whose decision changes when the game is replaced by its
                        additive projection, or by the game with its even part erased
    Every quantity is exactly zero for an additive game, except joint flips, which thresholds alone can create.
    """
    u = np.atleast_2d(u)
    pairs = pair_list(lat.n)
    ia = np.array([1 << a for a, b in pairs])
    ib = np.array([1 << b for a, b in pairs])
    u0 = u[:, [0]]
    ua, ub, uab = u[:, ia], u[:, ib], u[:, ia | ib]
    pos = u0 > 0
    same = lambda x: (x > 0) == pos
    joint = same(ua) & same(ub) & ~same(uab)
    add_pred = ua + ub - u0
    inter = ((uab > 0) != (add_pred > 0))
    _, odd, even = parity_parts(u, lat)
    mean = walsh(u)[:, [0]]
    add = additive_projection(u, lat)
    sign = u > 0
    return {"joint_flip_pairs": joint, "interaction_flip_pairs": inter,
            "joint_flip": joint.any(1), "interaction_flip": inter.any(1),
            "additive_decision_change": (sign != (add > 0)).mean(1),
            "even_erasure_decision_change": (sign != ((mean + odd) > 0)).mean(1)}


def images_needed(v, length, delta, target, n_max=10 ** 7):
    """Smallest N whose empirical Bernstein radius, at the observed sample variance v, falls below |target|."""
    target = abs(float(target))
    if not np.isfinite(target) or target <= 0:
        return np.inf
    lg = math.log(4.0 / delta)
    rad = lambda N: math.sqrt(2 * v * lg / N) + 7 * length * lg / (3 * (N - 1))
    if rad(n_max) >= target:
        return np.inf
    lo, hi = 2, n_max
    while lo < hi:
        mid = (lo + hi) // 2
        if rad(mid) < target:
            hi = mid
        else:
            lo = mid + 1
    return lo


def mean_certificate(x, lo, hi, delta):
    """Mean of bounded per-image values in [lo, hi] with Hoeffding and empirical Bernstein radii at level 1 - delta."""
    x = np.asarray(x, float)
    N = len(x)
    L = float(hi - lo)
    m = float(x.mean()) if N else np.nan
    v = float(x.var(ddof=1)) if N > 1 else np.nan
    rh = hoeffding(L, N, delta) if N else np.inf
    rb = empirical_bernstein(x, L, delta)
    return {"N": N, "mean": m, "sd": math.sqrt(v) if N > 1 else np.nan, "range_length": L,
            "hoeffding_radius": rh, "bernstein_radius": rb,
            "hoeffding_sign": int(np.sign(m)) if abs(m) > rh else 0,
            "bernstein_sign": int(np.sign(m)) if abs(m) > rb else 0,
            "images_needed_bernstein": images_needed(v, L, delta, m) if N > 1 else np.inf}

## Heads, removal games and populations

In [ ]:
# ============================================================
# HEADS, REMOVAL GAMES, POPULATIONS AND THE FROZEN INVERSE TEMPERATURE
# A hard removal game depends on an image only through its predicted states, so for vocabularies of at most
# 12 concepts every head is evaluated once on all 3^n masked combinations (state absent, state present, or the
# fill of a removed concept). Hard games are lookups in this table, the exact expected-hard game is one tensor
# contraction and the certified ranges of every linear per-image quantity are extremes over all 2^n profiles.
# Larger vocabularies evaluate the head directly on the 2^n masked inputs of each image. The fill is a zero
# block (zero removal, primary) or the training frequencies of the predicted states (marginal removal).
# Game values are malignancy logits s = logit_1 - logit_0, oriented by the predicted class in every population.
# ============================================================
import gc


class NumpyHead:
    """Float64 mirror of a trained diagnosis head, read from the specification written by Part 1."""

    def __init__(self, spec, n):
        self.spec = spec
        self.sizes = [2] * n

    def logit(self, v, start=0):
        h = np.asarray(v, float)
        for layer in self.spec[start:]:
            if layer[0] == "linear":
                h = h @ layer[1].T + layer[2]
            elif layer[0] == "layernorm":
                mu = h.mean(-1, keepdims=True)
                var = h.var(-1, keepdims=True)
                h = (h - mu) / np.sqrt(var + layer[3]) * layer[1] + layer[2]
            elif layer[0] == "relu":
                h = np.maximum(h, 0.0)
        return h[..., 1] - h[..., 0]


def make_head(kind, n, hidden=128):
    d, h = 2 * n, hidden
    if kind == "affine":
        return nn.Sequential(nn.Linear(d, 2))
    if kind == "mlp":
        return nn.Sequential(nn.Linear(d, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(0.3), nn.Linear(h, 2))
    layers, w = [], d
    for _ in range(3):
        layers += [nn.Linear(w, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(0.3)]
        w = h
    return nn.Sequential(*layers, nn.Linear(h, 2))


def torch_head(entry):
    head = make_head(entry["kind"], len(entry["columns"]), MANIFEST["training"].get("HEAD_HIDDEN", 128))
    head.load_state_dict(entry["state_dict"])
    head.eval()
    return head


class TorchLogit:
    """The deployed float32 head with the interface of NumpyHead (measures the table error of the mirror)."""

    def __init__(self, head, n):
        self.head = head
        self.sizes = [2] * n

    def logit(self, v):
        with torch.no_grad():
            o = self.head(torch.tensor(np.asarray(v), dtype=torch.float32))
        return (o[:, 1] - o[:, 0]).double().numpy()


def one_hot_blocks(hard):
    L, n = hard.shape
    X = np.zeros((L, 2 * n))
    X[np.arange(L)[:, None], 2 * np.arange(n)[None, :] + hard] = 1.0
    return X


def masked_inputs(sizes, fills):
    """Every head input with one state or the fill per block. Index sizes[j] on axis j is the fill of concept j."""
    grids = []
    for k, fill in zip(sizes, fills):
        blk = np.vstack([np.eye(k), np.asarray(fill, float)[None, :]])        # (k + 1, k)
        grids.append(blk)
    shape = tuple(k + 1 for k in sizes)
    idx = np.indices(shape).reshape(len(sizes), -1).T                           # C order, last axis fastest
    return np.concatenate([grids[j][idx[:, j]] for j in range(len(sizes))], 1), shape


def masked_table(ncbm, fills, batch=65536):
    X, shape = masked_inputs(ncbm.sizes, fills)
    out = np.concatenate([ncbm.logit(X[i:i + batch]) for i in range(0, len(X), batch)])
    return out.reshape(shape)


def _strides(shape):
    return np.array([int(np.prod(shape[j + 1:])) for j in range(len(shape))], dtype=np.int64)


def games_from_profiles(G, profiles, lat):
    """Hard removal games u(A) for every profile (rows) and subset A (columns), by lookup in G."""
    shape = G.shape
    st = _strides(shape)
    prof = np.atleast_2d(profiles).astype(np.int64)
    fill = np.array(shape) - 1
    base = prof @ st
    delta = (fill[None, :] - prof) * st[None, :]                                # move from state to fill
    idx = base[:, None] + delta @ lat.mem.T.astype(np.int64)                    # (L, 2^n)
    return G.ravel()[idx]


def expected_hard_games(G, probs, lat):
    """
    Exact expected-hard removal games under the product law of the concept heads. For each image, axis j of G is
    contracted with the 2 x (|V_j| + 1) matrix whose first row holds the state probabilities (concept kept) and
    whose second row selects the fill (concept removed).
    """
    n = lat.n
    L = probs[0].shape[0]
    shape = G.shape
    letters = "abcdefghijklmnop"[:n]
    caps = letters.upper()
    expr = letters + "," + ",".join(f"{caps[j]}{letters[j]}" for j in range(n)) + "->" + caps
    bitpos = np.array([sum(((A >> j) & 1) << (n - 1 - j) for j in range(n)) for A in range(lat.N)])
    out = np.zeros((L, lat.N))
    for l in range(L):
        Qs = []
        for j in range(n):
            q = np.zeros((2, shape[j]))
            q[0, :shape[j] - 1] = probs[j][l]
            q[1, shape[j] - 1] = 1.0
            Qs.append(q)
        T = np.einsum(expr, G, *Qs, optimize="greedy")
        out[l] = T.ravel()[bitpos]
    return out


def mc_expected_hard_game(G, prob_one, lat, M, rng, chunk=20000):
    """Monte Carlo expected-hard game of one image from M shared state draws (Proposition 2 iii)."""
    shape = G.shape
    st = _strides(shape)
    fill = np.array(shape) - 1
    n = lat.n
    acc = np.zeros(lat.N)
    done = 0
    memT = lat.mem.T.astype(np.int64)
    while done < M:
        m = min(chunk, M - done)
        prof = np.stack([rng.choice(shape[j] - 1, size=m, p=prob_one[j]) for j in range(n)], 1).astype(np.int64)
        base = prof @ st
        idx = base[:, None] + ((fill[None, :] - prof) * st[None, :]) @ memT
        acc += G.ravel()[idx].sum(0)
        done += m
    return acc / M


_ALLPROF = {}


def all_profiles(sizes):
    key = tuple(sizes)
    if key not in _ALLPROF:
        _ALLPROF[key] = np.indices(tuple(sizes)).reshape(len(sizes), -1).T
    return _ALLPROF[key]


def profile_ranges(G, sizes, lat):
    """
    Certified ranges of the per-image scalars over every predicted concept profile. Any hard game, hence any image,
    has values inside these ranges, and the expected-hard game is a convex combination of hard games, so the
    linear quantities (effects, interaction indices) of expected-hard games stay inside them as well.
    """
    U = games_from_profiles(G, all_profiles(sizes), lat)
    W = walsh_diagnostics(U, lat)
    return {"logit_min": float(G.min()), "logit_max": float(G.max()),
            "effects_min": W["effects"].min(0), "effects_max": W["effects"].max(0),
            "I_min": W["I"].min(0), "I_max": W["I"].max(0), "n_profiles": U.shape[0]}


def direct_games(ncbm, profiles, fills, lat, chunk=1 << 16):
    """Removal games by direct evaluation of the head on the 2^n masked inputs, once per distinct profile."""
    prof = np.atleast_2d(profiles)
    uniq, inv = np.unique(prof, axis=0, return_inverse=True)
    fillvec = np.concatenate([np.asarray(f, float) for f in fills])
    memrep = np.repeat(lat.mem, 2, axis=1)                                     # (2^n, 2n)
    games = np.zeros((len(uniq), lat.N))
    t0 = time.time()
    first = getattr(ncbm, "spec", None)
    if first is not None and first[0][0] == "linear":
        # the first layer is linear, so hiding concept j shifts its pre-activation by the fixed vector D_j and
        # the 2^n pre-activations are one matrix product instead of 2^n masked inputs
        W1, b1 = first[0][1], first[0][2]
        memf = lat.mem.astype(float)
        for i, p in enumerate(uniq):
            x = one_hot_blocks(p[None, :])[0]
            D = np.stack([(fillvec[2 * j:2 * j + 2] - x[2 * j:2 * j + 2]) @ W1[:, 2 * j:2 * j + 2].T
                          for j in range(lat.n)])                                # (n, hidden)
            h0 = x @ W1.T + b1
            for s in range(0, lat.N, chunk):
                games[i, s:s + chunk] = ncbm.logit(h0[None, :] + memf[s:s + chunk] @ D, start=1)
    else:
        for i, p in enumerate(uniq):
            x = one_hot_blocks(p[None, :])[0]
            for s in range(0, lat.N, chunk):
                X = np.where(memrep[s:s + chunk], fillvec[None, :], x[None, :])
                games[i, s:s + chunk] = ncbm.logit(X)
    if lat.n > TABLE_MAX_N:
        say(f"    {len(prof)} games ({len(uniq)} distinct profiles) with {lat.n} concepts: {time.time() - t0:.0f} s")
    return games[np.ravel(inv)]


TABLE_MAX_N = 12
RUNS = {}


def get_run(seed):
    """States, heads, masked tables and predictions of one seed (cached per file fingerprint)."""
    cpath = os.path.join(CFG.CACHE_DIR, f"run_seed{seed}_{model_stamp(seed)}.pkl")
    if os.path.exists(cpath):
        return _pkl_load(cpath)
    t0 = time.time()
    e = MANIFEST["seeds"][str(seed)]
    z = np.load(e["states"], allow_pickle=True)
    st = {k: z[k] for k in z.files}
    heads = torch.load(e["heads"], map_location="cpu", weights_only=False)
    rec = pd.read_csv(e["predictions"])
    run = {"seed": seed, "states": st, "heads": {}}
    for key in HEAD_KEYS:
        ent = heads[key]
        cols = ent["columns"]
        n = len(cols)
        nh = NumpyHead(ent["spec"], n)
        p = st["hard_train"][:, cols].mean(0)
        fills = {"zero": [np.zeros(2) for _ in range(n)], "marginal": [np.array([1 - q, q]) for q in p]}
        h = {"key": key, "kind": ent["kind"], "vocab": ent["vocab"], "columns": cols, "n": n, "ncbm": nh,
             "fills": fills, "tables": {}, "pred": {}}
        say(f"  seed {seed}, head {key}: {'masked tables' if n <= TABLE_MAX_N else 'predictions'}")
        if n <= TABLE_MAX_N:
            th = TorchLogit(torch_head(ent), n)
            for r in CFG.REMOVALS:
                h["tables"][r] = masked_table(nh, fills[r])
                h["tables"][r + "32"] = masked_table(th, fills[r])
        for s in ("valid", "test"):
            logit = nh.logit(one_hot_blocks(st[f"hard_{s}"][:, cols]))
            r = rec[(rec["head"] == key) & (rec["split"] == s)]
            same = np.array_equal(r["image_id"].astype(str).values, st[f"id_{s}"].astype(str))
            h["pred"][s] = {"logit": logit, "correct": (logit > 0).astype(int) == st[f"y_{s}"],
                            "agreement_with_part1": float(((logit > 0).astype(int) == r["y_pred"].values).mean())
                            if same else np.nan,
                            "max_logit_gap_part1": float(np.abs(logit - r["logit"].values).max()) if same else np.nan}
        run["heads"][key] = h
    run["seconds"] = time.time() - t0
    _pkl_save(run, cpath)
    return run


def head_games(h, removal, profiles):
    lat = lattice(h["n"])
    if removal in h["tables"]:
        return games_from_profiles(h["tables"][removal], profiles, lat)
    return direct_games(h["ncbm"], profiles, h["fills"][removal], lat)


def table_error(h, removal, profiles):
    """Sup-norm gap between the float64 mirror and the deployed float32 head on each image's game."""
    lat = lattice(h["n"])
    if removal + "32" in h["tables"]:
        g64 = games_from_profiles(h["tables"][removal], profiles, lat)
        g32 = games_from_profiles(h["tables"][removal + "32"], profiles, lat)
        return np.abs(g64 - g32).max(1) + CFG.NUMERICAL_FLOOR
    return np.full(len(profiles), np.nan)


_PG_MEMO = {}


def population_games(run, key, removal, pop, split="test"):
    """
    Oriented games (u(empty set) > 0 for every correctly classified image) of one population. Heads with more than
    12 concepts keep a fixed random draw of CFG.LARGE_N_IMAGES images per population and split, whose test games
    are memoized because Experiments 1 and 3 reuse them.
    """
    mk = (run["seed"], key, removal, pop, split)
    if mk in _PG_MEMO:
        return _PG_MEMO[mk]
    h = run["heads"][key]
    st = run["states"]
    cls = CFG.POPULATIONS[pop]
    keep = h["pred"][split]["correct"] & (st[f"y_{split}"] == cls)
    if h["n"] > TABLE_MAX_N and keep.sum() > CFG.LARGE_N_IMAGES:
        idx = np.where(keep)[0]
        draw = np.random.default_rng([run["seed"], cls, int(split == "test")]).choice(idx, CFG.LARGE_N_IMAGES, replace=False)
        keep = np.zeros_like(keep)
        keep[draw] = True
    prof = st[f"hard_{split}"][keep][:, h["columns"]]
    sign = 1.0 if cls == 1 else -1.0
    out = {"keep": keep, "profiles": prof, "sign": sign, "N": int(keep.sum()),
           "ids": st[f"id_{split}"][keep], "groups": st[f"group_{split}"][keep]}
    if removal == "expected_zero":
        probs = [np.stack([1 - st[f"prob_{split}"][keep][:, c], st[f"prob_{split}"][keep][:, c]], 1)
                 for c in h["columns"]]
        u = expected_hard_games(h["tables"]["zero"], probs, lattice(h["n"])) if keep.sum() else np.zeros((0, 1 << h["n"]))
        out["eps"] = np.full(int(keep.sum()), float(np.abs(h["tables"]["zero"] - h["tables"]["zero32"]).max())
                             + CFG.NUMERICAL_FLOOR)
    else:
        u = head_games(h, removal, prof) if keep.sum() else np.zeros((0, 1 << h["n"]))
        out["eps"] = table_error(h, removal, prof) if keep.sum() else np.zeros(0)
    out["u"] = sign * u
    if h["n"] > TABLE_MAX_N and split == "test":
        _PG_MEMO[mk] = out
    return out


def oriented_ranges(run, key, removal, sign):
    """Certified ranges of effects and pairwise Banzhaf interactions over every hard profile, in class orientation."""
    h = run["heads"][key]
    tab = h["tables"]["zero" if removal == "expected_zero" else removal]
    return profile_ranges(sign * tab, [2] * h["n"], lattice(h["n"]))


# ---------------- inverse temperature, frozen on validation images ----------------

def _group_halves(groups, seed):
    g = np.unique(groups)
    first = set(np.random.default_rng(seed).permutation(g)[: len(g) // 2])
    return np.array([x in first for x in groups])


def beta0_curve(run, key, s):
    """Split-half agreement of the local pair shares between two halves of duplicate groups, per beta0."""
    rows = []
    halves = {}
    for pop in CFG.POPULATIONS:
        pg = population_games(run, key, "zero", pop, split="valid")
        if pg["N"] >= 4:
            halves[pop] = (pg["u"], _group_halves(pg["groups"], run["seed"]))
    lat = lattice(run["heads"][key]["n"])
    iu = np.triu_indices(lat.n, 1)
    for b0 in CFG.BETA0_GRID:
        agree = []
        for pop, (u, half) in halves.items():
            if half.sum() < 2 or (~half).sum() < 2:
                continue
            for A, B in ((half, ~half), (~half, half)):
                M = gibbs_sofi_dp(u[A], b0 / s, lat, want_chain=False, chunk=256)["pair"].mean(0)
                p = gibbs_sofi_dp(u[B], b0 / s, lat, want_chain=False, chunk=256)["pair"][:, iu[0], iu[1]]
                q = np.broadcast_to(M[iu], p.shape)
                agree.append(1 - ((q - p) ** 2).sum() / max(((0.5 - p) ** 2).sum(), 1e-300))
        rows.append({"seed": run["seed"], "beta0": b0, "beta": b0 / s,
                     "split_half_agreement": np.mean(agree) if agree else np.nan, "comparisons": len(agree)})
    return pd.DataFrame(rows)


def score_scale(run, key, rng):
    """Median null standard deviation of the score over the correctly classified validation images."""
    us = [population_games(run, key, "zero", pop, split="valid")["u"] for pop in CFG.POPULATIONS]
    u = np.concatenate([x for x in us if len(x)], 0)
    sd, how = null_sd(u, lattice(run["heads"][key]["n"]), rng, CFG.NULL_MC_DRAWS)
    med = float(np.median(sd[sd > 0])) if (sd > 0).any() else 1.0
    return med, how


class LazyScales(dict):
    """
    Null scales per head. Heads on V12 are computed for every seed, while the larger heads, which only the seed
    studied in depth uses, are computed on first access (each of their games needs up to 2^20 head evaluations).
    """

    def __init__(self, run, rng, methods):
        super().__init__()
        self.run, self.rng, self.methods = run, rng, methods

    def __missing__(self, key):
        seed = self.run["seed"]
        say(f"  seed {seed}, head {key}: null scale of the score")
        val, how = cached(f"scale_{key}_seed{seed}_{model_stamp(seed)}", lambda: score_scale(self.run, key, self.rng))
        self[key] = val
        self.methods[key] = how
        return val


def frozen_beta(run):
    """beta0 of one seed, chosen on validation images with the primary head, and the scale of every head."""
    seed = run["seed"]
    key0 = f"beta0_seed{seed}_{model_stamp(seed)}"
    rng = np.random.default_rng(seed)
    methods = {}
    scales = LazyScales(run, rng, methods)
    for key in HEAD_KEYS:
        if run["heads"][key]["n"] <= TABLE_MAX_N:
            scales[key]
    curve = cached(f"beta0_curve_seed{seed}_{model_stamp(seed)}",
                   lambda: beta0_curve(run, CFG.PRIMARY_HEAD, scales[CFG.PRIMARY_HEAD]))
    fr = frozen_choices()
    if key0 in fr:
        beta0 = fr[key0]
    elif curve["split_half_agreement"].notna().any():
        best = curve["split_half_agreement"].round(6).max()
        beta0 = float(curve[curve["split_half_agreement"].round(6) == best]["beta0"].min())
    else:
        beta0 = 1.0
    save_frozen(**{key0: beta0})
    curve = curve.assign(chosen=np.isclose(curve["beta0"], beta0))
    return beta0, scales, methods, curve


banner("Loading trained heads and building the masked tables")
BETA = {}
for seed in CFG.SEEDS:
    t0 = time.time()
    RUNS[seed] = get_run(seed)
    b0, scales, methods, curve = frozen_beta(RUNS[seed])
    BETA[seed] = {"beta0": b0, "scale": scales, "scale_method": methods}
    show(curve, f"Split-half agreement of local pair shares, seed {seed}", key="tuning_beta0")
    rows = []
    for key, h in RUNS[seed]["heads"].items():
        for s in ("valid", "test"):
            y = RUNS[seed]["states"][f"y_{s}"]
            c = h["pred"][s]["correct"]
            rows.append({"seed": seed, "head": key, "split": s, "images": len(y),
                         "correct_malignant": int((c & (y == 1)).sum()), "correct_non_malignant": int((c & (y == 0)).sum()),
                         "balanced_accuracy": 0.5 * (c[y == 1].mean() + c[y == 0].mean()),
                         "agreement_with_part1": h["pred"][s]["agreement_with_part1"],
                         "max_logit_gap_part1": h["pred"][s]["max_logit_gap_part1"],
                         "scale_s": dict.get(BETA[seed]["scale"], key, np.nan),
                         "scale_method": BETA[seed]["scale_method"].get(key, "computed when first used")})
    show(pd.DataFrame(rows), f"Heads of seed {seed} (beta0 = {b0:g})", key="heads")
    say(f"seed {seed}: {time.time() - t0:.0f} s")

_sel = pd.concat(TABLES["heads"])
_sel = _sel[(_sel["head"] == CFG.PRIMARY_HEAD) & (_sel["split"] == "valid")]
SEL = int(_sel.sort_values(["balanced_accuracy", "seed"], ascending=[False, True]).iloc[0]["seed"])
SEL = frozen_choices().get("selected_seed", SEL)
save_frozen(selected_seed=SEL)
say(f"\nSeed studied in depth (highest validation balanced accuracy of the primary head): {SEL}")

## Experiment 1. Computation of local ranking laws

Identities of the implementation, lazy Metropolis and replica exchange against the exact laws for 12, 16 and 20 concepts (with the exact i.i.d. floor at the same number of draws and the R-hat verdict), and the Monte Carlo expected-hard game against its exact contraction.

In [ ]:
# ============================================================
# EXPERIMENT 1. COMPUTATION OF LOCAL RANKING LAWS
# 1a identities of the implementation (lookup against direct evaluation, float32 table error, Banzhaf
#    interactions from the Walsh transform against the definition, forward against backward recursions)
# 1b samplers against exact laws for 12, 16 and 20 concepts, with the exact i.i.d. floor at the same number of
#    recorded draws and the R-hat verdict
# 1c the Monte Carlo expected-hard game against its exact tensor contraction
# ============================================================

def exp1_identities():
    rows = []
    rng = np.random.default_rng(0)
    for seed in CFG.SEEDS:
        run = RUNS[seed]
        for key in HEAD_KEYS:
            h = run["heads"][key]
            lat = lattice(h["n"])
            prof = run["states"]["hard_test"][:, h["columns"]]
            idx = rng.choice(len(prof), min(CFG.IDENTITY_IMAGES if h["n"] <= 12 else 3, len(prof)), replace=False)
            for r in CFG.REMOVALS:
                d = direct_games(h["ncbm"], prof[idx], h["fills"][r], lat)
                g = head_games(h, r, prof[idx])
                W = walsh_diagnostics(d, lat)
                dp = gibbs_sofi_dp(d[:5], BETA[seed]["beta0"] / BETA[seed]["scale"][key], lat, want_chain=False) \
                    if h["n"] <= 12 else None
                rows.append({"seed": seed, "head": key, "removal": r, "images": len(idx),
                             "lookup_vs_direct": float(np.abs(g - d).max()),
                             "float32_table_error": float(table_error(h, r, prof[idx]).max()) if h["n"] <= 12 else np.nan,
                             "banzhaf_walsh_vs_definition": float(np.abs(W["I"] - banzhaf_interaction_brute(d, lat)).max())
                             if h["n"] <= 12 else np.nan,
                             "logZ_forward_vs_backward": float(np.abs(dp["logZ"] - dp["logZ_forward"]).max())
                             if dp is not None else np.nan,
                             "affine_NA_max": float(W["NA"].max()) if h["kind"] == "affine" else np.nan})
    return pd.DataFrame(rows)


def _sampler_images(u, beta, lat, k):
    """k images at evenly spaced quantiles of beta * osc, the quantity of the mixing bound."""
    eta = u[:, lat.comp] - u
    inner = lat.interior
    osc = eta[:, inner].max(1) - eta[:, inner].min(1)
    order = np.argsort(beta * osc)
    pick = order[np.unique(np.round(np.linspace(0, len(order) - 1, k)).astype(int))]
    return pick, osc


def exp1_sampler():
    rows = []
    seed = SEL
    run = RUNS[seed]
    rng = np.random.default_rng(seed)
    for key in CFG.SCALE_HEADS:
        h = run["heads"][key]
        lat = lattice(h["n"])
        beta = BETA[seed]["beta0"] / BETA[seed]["scale"][key]
        for pop in CFG.POPULATIONS:
            pg = population_games(run, key, "zero", pop)
            if pg["N"] < 2:
                continue
            pick, osc = _sampler_images(pg["u"], beta, lat, CFG.MCMC_IMAGES)
            for i in pick:
                u = pg["u"][i]
                t0 = time.time()
                ex = gibbs_single(u, beta, lat)
                t_exact = time.time() - t0
                eta = (u[lat.comp] - u)[None, :]
                eta[:, 0] = 0.0
                eta[:, lat.full] = 0.0
                for smp, ladder in (("metropolis", (1.0,)), ("replica_exchange", tuple(CFG.RE_LADDER))):
                    t0 = time.time()
                    snap_at = sorted(set(CFG.MCMC_CHECKPOINTS) | {t // 2 for t in CFG.MCMC_CHECKPOINTS})
                    snaps, rec, swap = run_chains(eta, np.array([beta]), [0] * CFG.MCMC_CHAINS, CFG.MCMC_STEPS,
                                                  CFG.MCMC_RECORD_EVERY, snap_at, rng, ladder=ladder,
                                                  swap_every=CFG.RE_SWAP_EVERY, pairs=True)
                    t_mcmc = time.time() - t0
                    for t in CFG.MCMC_CHECKPOINTS:
                        # second half of the recorded states of every chain, as in the controlled benchmark
                        Pc = snaps[t][0] - snaps[t // 2][0]
                        Pr = snaps[t][1] - snaps[t // 2][1]
                        draws = int(Pc.sum(0)[:, 0].sum())
                        P = Pc.sum(0) / draws
                        pr = Pr.sum(0) / draws
                        iid = exact_samples(u, beta, lat, draws, rng)
                        Pi, pri = position_counts(iid, lat.n)
                        r0, r1 = (t // 2) // CFG.MCMC_RECORD_EVERY, t // CFG.MCMC_RECORD_EVERY
                        rows.append({"seed": seed, "head": key, "n": lat.n, "population": pop, "image": int(i),
                                     "sampler": smp, "steps": t, "draws": draws, "beta_osc": float(beta * osc[i]),
                                     "max_error_P": float(np.abs(P - ex["P"]).max()),
                                     "max_error_P_iid": float(np.abs(Pi / draws - ex["P"]).max()),
                                     "max_error_pair": float(np.abs(pr - ex["pair"]).max()),
                                     "max_error_pair_iid": float(np.abs(pri / draws - ex["pair"]).max()),
                                     "rhat_rank": rhat_rank(rec[r0:r1]) if r1 - r0 >= 8 else np.nan,
                                     "ess_bulk": ess_bulk(rec[r0:r1]) if r1 - r0 >= 8 else np.nan,
                                     "log10_mixing_bound": mixing_time_bound_log10(beta, osc[i], lat.n, CFG.MIX_THETA),
                                     "swap_acceptance": swap, "seconds_exact": t_exact,
                                     "seconds_sampler_total": t_mcmc})
            say(f"  sampler study {key} {pop}: done")
    return pd.DataFrame(rows)


def exp1_expected_hard():
    rows = []
    seed = SEL
    run = RUNS[seed]
    h = run["heads"][CFG.PRIMARY_HEAD]
    lat = lattice(h["n"])
    rng = np.random.default_rng(seed + 1)
    s = BETA[seed]["scale"][CFG.PRIMARY_HEAD]
    for pop in CFG.POPULATIONS:
        pg = population_games(run, CFG.PRIMARY_HEAD, "expected_zero", pop)
        if pg["N"] == 0:
            continue
        pick = rng.choice(pg["N"], min(CFG.EH_IMAGES, pg["N"]), replace=False)
        keep = np.where(pg["keep"])[0][pick]
        for i, row in zip(pick, keep):
            prob_one = [np.array([1 - run["states"]["prob_test"][row, c], run["states"]["prob_test"][row, c]])
                        for c in h["columns"]]
            exact = pg["u"][i]
            for M in CFG.EH_DRAWS:
                mc = pg["sign"] * mc_expected_hard_game(h["tables"]["zero"], prob_one, lat, M, rng)
                We, Wm = walsh_diagnostics(exact[None], lat), walsh_diagnostics(mc[None], lat)
                rows.append({"seed": seed, "population": pop, "image": int(i), "draws": M,
                             "sup_error_over_s": float(np.abs(mc - exact).max() / s),
                             "banzhaf_interaction_error_over_s": float(np.abs(We["I"] - Wm["I"]).max() / s),
                             "hoeffding_sup_bound_over_s": float((h["tables"]["zero"].max() - h["tables"]["zero"].min())
                                                                 * math.sqrt(math.log(2 * lat.N / 0.05) / (2 * M)) / s)})
    return pd.DataFrame(rows)


banner("EXPERIMENT 1. COMPUTATION OF LOCAL RANKING LAWS")
E1_ID = cached(f"e1_identities_{'_'.join(model_stamp(s) for s in CFG.SEEDS)}", exp1_identities)
show(E1_ID, "1a. Identities of the implementation", key="e1_identities")
E1_S = cached(f"e1_sampler_seed{SEL}_{model_stamp(SEL)}", exp1_sampler)
show(E1_S, "1b. Samplers against exact laws (image level)", key="e1_sampler", echo=False)
_last = E1_S[E1_S.steps == max(CFG.MCMC_CHECKPOINTS)].copy()
_last["ratio"] = _last["max_error_P"] / _last["max_error_P_iid"]
_last["rhat_pass"] = _last["rhat_rank"] < 1.01
_last["inaccurate_pass"] = _last["rhat_pass"] & (_last["ratio"] > 2)
show(_last.groupby(["n", "sampler"]).agg(images=("image", "size"), median_beta_osc=("beta_osc", "median"),
                                         median_error=("max_error_P", "median"),
                                         median_iid_floor=("max_error_P_iid", "median"),
                                         median_ratio=("ratio", "median"), rhat_passes=("rhat_pass", "sum"),
                                         inaccurate_passes=("inaccurate_pass", "sum"),
                                         median_log10_bound=("log10_mixing_bound", "median"),
                                         seconds_exact=("seconds_exact", "median"),
                                         seconds_sampler=("seconds_sampler_total", "median")).reset_index(),
     f"1b. Samplers at {max(CFG.MCMC_CHECKPOINTS)} steps per chain (ratio = error / exact i.i.d. floor)",
     key="e1_sampler_summary")
E1_EH = cached(f"e1_expected_hard_seed{SEL}_{model_stamp(SEL)}", exp1_expected_hard)
show(E1_EH.groupby(["population", "draws"])[["sup_error_over_s", "banzhaf_interaction_error_over_s",
                                            "hoeffding_sup_bound_over_s"]].median().reset_index(),
     "1c. Monte Carlo expected-hard game against the exact contraction (medians over images)", key="e1_expected_hard")

## Experiment 2. Interactions

Head controls (affine, published two-layer, deeper), certificates for population means with Hoeffding and empirical Bernstein radii, parity duality on real games, decision-level magnitude and the pairwise indices of other methods.

In [ ]:
# ============================================================
# EXPERIMENT 2. INTERACTIONS
# 2a head controls: the affine head must give NA = 0 on every image, the deeper head should raise NA
# 2b certificates for population means (pairwise Banzhaf interactions, NA and its even share), with Hoeffding
#    and empirical Bernstein radii over certified ranges and the images each unresolved sign would need
# 2c parity duality on real games: erasing or reversing the even part leaves every ranking law unchanged
# 2d decision-level magnitude: flips that need the second difference, and decisions changed by the even part
# 2e pairwise indices of other methods on the same games, with the odd leakage of Shapley-Taylor
# Every configuration is one seed, one head, one game and one population of correctly classified test images.
# ============================================================
GAMES_FOR = {k: list(CFG.REMOVALS) for k in CFG.CONTROL_HEADS}
GAMES_FOR[CFG.PRIMARY_HEAD] = list(CFG.REMOVALS) + ["expected_zero"]
PG = {}


def dmax_chunked(u, lat, chunk=48):
    return {"dmax": np.concatenate([deg_extremes_dp(u[i:i + chunk], lat)["dmax"] for i in range(0, len(u), chunk)])}


def pg(seed, key, game, pop):
    k = (seed, key, game, pop)
    if k not in PG:
        PG[k] = population_games(RUNS[seed], key, game, pop)
    return PG[k]


def exp2_controls_and_certificates():
    ctrl, cert, dec, pairdec = [], [], [], []
    for seed in CFG.SEEDS:
        run = RUNS[seed]
        for key in CFG.CONTROL_HEADS:
            h = run["heads"][key]
            lat = lattice(h["n"])
            pairs = pair_list(h["n"])
            names = [f"{short(CONCEPTS[h['columns'][a]])} / {short(CONCEPTS[h['columns'][b]])}" for a, b in pairs]
            for game in GAMES_FOR[key]:
                for pop in CFG.POPULATIONS:
                    P = pg(seed, key, game, pop)
                    if P["N"] < CFG.MIN_POPULATION_IMAGES:
                        continue
                    u = P["u"]
                    W = walsh_diagnostics(u, lat)
                    cert_na, lo, hi = certified_na(W, P["eps"])
                    # the even share is defined only where non-additivity is certified (affine heads have NA at rounding level)
                    even_share = np.where(cert_na, W["NA_even"] / np.where(W["NA"] > 0, W["NA"], 1), np.nan)
                    has_na = cert_na.sum() >= 2
                    D = decision_diagnostics(u, lat)
                    base = {"seed": seed, "head": key, "game": game, "population": pop, "N": P["N"]}
                    ctrl.append({**base, "NA_median": float(np.median(W["NA"])), "NA_mean": float(W["NA"].mean()),
                                 "NA_max": float(W["NA"].max()), "certified_non_additive": float(cert_na.mean()),
                                 "NA_lower_median": float(np.median(lo)),
                                 "even_share_mean": float(np.nanmean(even_share)) if cert_na.any() else np.nan,
                                 "odd_share_mean": float(1 - np.nanmean(even_share)) if cert_na.any() else np.nan,
                                 "max_table_error": float(np.nanmax(P["eps"]))})
                    dec.append({**base, "joint_flip_images": float(D["joint_flip"].mean()),
                                "interaction_flip_images": float(D["interaction_flip"].mean()),
                                "additive_decision_change_mean": float(D["additive_decision_change"].mean()),
                                "additive_decision_change_any": float((D["additive_decision_change"] > 0).mean()),
                                "even_erasure_decision_change_mean": float(D["even_erasure_decision_change"].mean()),
                                "even_erasure_decision_change_any": float((D["even_erasure_decision_change"] > 0).mean())})
                    freq = D["interaction_flip_pairs"].mean(0)
                    for i in np.argsort(-freq)[:5]:
                        if freq[i] > 0:
                            pairdec.append({**base, "pair": names[i], "interaction_flip_share": float(freq[i]),
                                            "joint_flip_share": float(D["joint_flip_pairs"][:, i].mean())})
                    # certificates, delta split over the pairs and the two scalar means
                    rngs = oriented_ranges(run, key, game, P["sign"])
                    dl = CFG.DELTA / (len(pairs) + 2)
                    for i, nm in enumerate(names):
                        c = mean_certificate(W["I"][:, i], rngs["I_min"][i], rngs["I_max"][i], dl)
                        cert.append({**base, "quantity": "banzhaf_interaction", "pair": nm, **c})
                    cert.append({**base, "quantity": "NA", "pair": "", **mean_certificate(W["NA"], 0.0, 1.0, dl)})
                    if has_na:
                        x = even_share[cert_na] - 0.5
                        cert.append({**base, "quantity": "even_share_minus_half", "pair": "",
                                     **mean_certificate(x, -0.5, 0.5, dl)})
    return pd.DataFrame(ctrl), pd.DataFrame(cert), pd.DataFrame(dec), pd.DataFrame(pairdec)


def exp2_invisibility():
    """Proposition 6 on real games: the even part is invisible to every ranking law and to first-order values."""
    rows = []
    for seed in CFG.SEEDS:
        run = RUNS[seed]
        for key in CFG.CONTROL_HEADS:
            h = run["heads"][key]
            lat = lattice(h["n"])
            beta = BETA[seed]["beta0"] / BETA[seed]["scale"][key]
            for pop in CFG.POPULATIONS:
                P = pg(seed, key, "zero", pop)
                if P["N"] < CFG.MIN_POPULATION_IMAGES:
                    continue
                u = P["u"]
                mean, odd, even = parity_parts(u, lat)
                twins = {"even part erased": mean[:, None] + odd, "even part reversed": mean[:, None] + odd - even}
                base = gibbs_sofi_dp(u, beta, lat, want_chain=False, chunk=256)
                Wb = walsh_diagnostics(u, lat)
                sh = shapley_values(u, lat)
                ex = dmax_chunked(u, lat)
                scale_I = np.abs(Wb["I"]).max(1)
                for tw, v in twins.items():
                    d = gibbs_sofi_dp(v, beta, lat, want_chain=False, chunk=256)
                    Wv = walsh_diagnostics(v, lat)
                    exv = dmax_chunked(v, lat)
                    rel = np.where(scale_I > 0, np.abs(Wv["I"] - Wb["I"]).max(1) / np.where(scale_I > 0, scale_I, 1), 0)
                    rows.append({"seed": seed, "head": key, "population": pop, "twin": tw, "N": P["N"],
                                 "soft_ranking_change": float(np.abs(d["P"] - base["P"]).max()),
                                 "pair_share_change": float(np.abs(d["pair"] - base["pair"]).max()),
                                 "sofi_optimum_change": float(np.abs(exv["dmax"] - ex["dmax"]).max()),
                                 "shapley_change": float(np.abs(shapley_values(v, lat) - sh).max()),
                                 "banzhaf_value_change": float(np.abs(Wv["effects"] - Wb["effects"]).max()),
                                 "banzhaf_interaction_relative_change_median": float(np.median(rel))})
    return pd.DataFrame(rows)


def exp2_indices():
    """Top pairs of every pairwise index, agreement with the Banzhaf interaction and odd leakage of Shapley-Taylor."""
    rows = []
    for seed in CFG.SEEDS:
        run = RUNS[seed]
        for key in CFG.CONTROL_HEADS:
            if key.endswith("affine"):
                continue
            h = run["heads"][key]
            lat = lattice(h["n"])
            for pop in CFG.POPULATIONS:
                P = pg(seed, key, "zero", pop)
                if P["N"] < CFG.MIN_POPULATION_IMAGES:
                    continue
                u = P["u"]
                idx = interaction_indices(u, lat)
                _, odd, even = parity_parts(u, lat)
                st_odd = interaction_indices(odd, lat)["shapley_taylor"]
                top_b = np.abs(idx["banzhaf"]).argmax(1)
                for m, v in idx.items():
                    rho = [sps.spearmanr(v[i], idx["banzhaf"][i])[0] if np.std(v[i]) > 0 and np.std(idx["banzhaf"][i]) > 0
                           else np.nan for i in range(len(u))]
                    rows.append({"seed": seed, "head": key, "population": pop, "method": m, "N": P["N"],
                                 "top_pair_agrees_with_banzhaf": float((np.abs(v).argmax(1) == top_b).mean()),
                                 "median_spearman_with_banzhaf": float(np.nanmedian(rho)),
                                 "odd_part_norm_share": float(np.median(
                                     np.linalg.norm(st_odd, axis=1) / np.maximum(np.linalg.norm(v, axis=1), 1e-300)))
                                 if m == "shapley_taylor" else np.nan})
    return pd.DataFrame(rows)


banner("EXPERIMENT 2. INTERACTIONS")
_stamp = "_".join(model_stamp(s) for s in CFG.SEEDS)
E2_CTRL, E2_CERT, E2_DEC, E2_PAIRDEC = cached(f"e2_main_{_stamp}", exp2_controls_and_certificates)
show(E2_CTRL, "2a. Head controls and non-additivity (all seeds, heads, games, populations)", key="e2_controls")
show(E2_CTRL.groupby(["head", "game", "population"])[["NA_median", "NA_max", "certified_non_additive",
                                                      "even_share_mean"]].agg(["mean", "min", "max"]).reset_index(),
     "2a. Head controls, range over seeds", key="e2_controls_summary")
show(E2_CERT, "2b. Certificates for population means", key="e2_certificates", echo=False)
_c = E2_CERT[E2_CERT["quantity"] == "banzhaf_interaction"]
show(_c.groupby(["head", "game", "population", "seed"]).agg(
    pairs=("pair", "size"), bernstein_certified=("bernstein_sign", lambda x: int((x != 0).sum())),
    hoeffding_certified=("hoeffding_sign", lambda x: int((x != 0).sum())),
    median_bernstein_radius=("bernstein_radius", "median"), median_hoeffding_radius=("hoeffding_radius", "median"),
    median_images_needed=("images_needed_bernstein", "median")).reset_index(),
     "2b. Certified signs of pairwise Banzhaf interactions per configuration", key="e2_certificates_summary")
show(E2_CERT[E2_CERT["quantity"] != "banzhaf_interaction"], "2b. Certificates for mean NA and for the even share",
     key="e2_scalar_certificates")
show(E2_DEC, "2d. Decision-level magnitude", key="e2_decisions")
show(E2_PAIRDEC, "2d. Pairs with the most interaction flips", key="e2_decision_pairs", echo=False)
E2_INV = cached(f"e2_invisibility_{_stamp}", exp2_invisibility)
show(E2_INV.groupby(["head", "twin"])[["soft_ranking_change", "pair_share_change", "sofi_optimum_change",
                                       "shapley_change", "banzhaf_value_change",
                                       "banzhaf_interaction_relative_change_median"]].max().reset_index(),
     "2c. Parity duality on real games (maximum over seeds and populations)", key="e2_invisibility")
TABLES.setdefault("e2_invisibility_full", []).append(E2_INV)
E2_IDX = cached(f"e2_indices_{_stamp}", exp2_indices)
show(E2_IDX.groupby(["head", "population", "method"])[["top_pair_agrees_with_banzhaf", "median_spearman_with_banzhaf",
                                                        "odd_part_norm_share"]].mean().reset_index(),
     "2e. Pairwise indices of other methods (mean over seeds)", key="e2_indices")
TABLES.setdefault("e2_indices_full", []).append(E2_IDX)

## Experiment 3. Compact summaries

Exact maximum-entropy reconstructions from the soft ranking and from the average positions, for 12, 16 and 20 concepts.

In [ ]:
# ============================================================
# EXPERIMENT 3. COMPACT SUMMARIES WITHOUT ENUMERATING ORDERINGS
# Maximum-entropy laws with the soft ranking (n^2 moments) or with the average positions (n moments) of the
# local ranking law, fitted exactly by the edge-weighted recursions. The divergence from the local law is exact,
# KL(mu || fit) = -H(mu) - <theta, P_mu> + log Z_theta, so no ordering is ever enumerated. The same fits run for
# 16 and 20 concepts on the images of the sampler study.
# ============================================================

def exp3_summaries():
    rows = []
    seed = SEL
    run = RUNS[seed]
    rng = np.random.default_rng(seed + 3)
    for key in CFG.SCALE_HEADS:
        h = run["heads"][key]
        lat = lattice(h["n"])
        beta = BETA[seed]["beta0"] / BETA[seed]["scale"][key]
        for pop in CFG.POPULATIONS:
            P = pg(seed, key, "zero", pop)
            if P["N"] < 2:
                continue
            if h["n"] <= 12:
                pick = np.arange(P["N"]) if P["N"] <= CFG.MAXENT_MAX_IMAGES else \
                    np.sort(rng.choice(P["N"], CFG.MAXENT_MAX_IMAGES, replace=False))
            else:
                pick3, _ = _sampler_images(P["u"], beta, lat, 3)
                pick = pick3[len(pick3) // 2: len(pick3) // 2 + CFG.MAXENT_LARGE_IMAGES]
            W = walsh_diagnostics(P["u"][pick], lat)
            for j, i in enumerate(pick):
                mu = gibbs_single(P["u"][i], beta, lat)
                rho = weyl_rho(lat.n)
                t0 = time.time()
                big = lat.n > 12                 # each recursion over 2^16 or 2^20 subsets takes seconds
                tol, it = (CFG.MAXENT_TOL_LARGE, CFG.MAXENT_MAXITER_LARGE) if big else (CFG.MAXENT_TOL, 5000)
                fP = maxent_positions(mu["P"], lat, kind="P", tol=tol, maxiter=it)
                fy = maxent_positions(mu["P"] @ rho, lat, kind="y", tol=tol, maxiter=it)
                rows.append({"seed": seed, "head": key, "n": lat.n, "population": pop, "image": int(i),
                             "entropy_share": mu["H"] / math.lgamma(lat.n + 1), "NA": float(W["NA"][j]),
                             "kl_soft_ranking_fit": fP["kl"](mu["P"], mu["H"]),
                             "kl_average_position_fit": fy["kl"](mu["P"], mu["H"]),
                             "pair_error_soft_ranking_fit": float(np.abs(fP["pair"] - mu["pair"]).max()),
                             "pair_error_average_position_fit": float(np.abs(fy["pair"] - mu["pair"]).max()),
                             "moment_residual_P": fP["resid"], "moment_residual_y": fy["resid"],
                             "iterations_P": fP["iterations"], "iterations_y": fy["iterations"],
                             "seconds": time.time() - t0})
        say(f"  summaries {key}: done")
    return pd.DataFrame(rows)


banner("EXPERIMENT 3. COMPACT SUMMARIES")
E3 = cached(f"e3_summaries_seed{SEL}_{model_stamp(SEL)}", exp3_summaries)
show(E3, "3. Maximum-entropy summaries (image level)", key="e3_summaries", echo=False)
_e3 = E3.copy()
_e3["NA_tercile"] = E3.groupby(["head", "population"])["NA"].transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["low", "middle", "high"]) if len(x) >= 3 else "all")
show(_e3.groupby(["n", "population"]).agg(images=("image", "size"),
                                         kl_P_median=("kl_soft_ranking_fit", "median"),
                                         kl_P_max=("kl_soft_ranking_fit", "max"),
                                         kl_y_median=("kl_average_position_fit", "median"),
                                         pair_error_P_median=("pair_error_soft_ranking_fit", "median"),
                                         pair_error_y_median=("pair_error_average_position_fit", "median"),
                                         max_residual_P=("moment_residual_P", "max"),
                                         seconds_median=("seconds", "median")).reset_index(),
     "3. Summaries per vocabulary and population", key="e3_summary")
show(_e3[_e3.n == 12].groupby(["population", "NA_tercile"])[["NA", "kl_soft_ranking_fit",
                                                            "pair_error_soft_ranking_fit"]].median().reset_index(),
     "3. Soft-ranking summaries against non-additivity (terciles of NA, 12 concepts)", key="e3_by_na")

## Experiment 4. Persistence

Only findings certified with the same sign in every seed under both removal rules are reported.

In [ ]:
# ============================================================
# EXPERIMENT 4. PERSISTENCE ACROSS SEEDS AND REMOVAL RULES
# A finding is reported only when it holds in every seed under both removal rules. For each head and population,
# a pairwise interaction is persistent when its empirical Bernstein certificate gives the same sign in all
# configurations. The same rule applies to certified non-additivity and to the sign of the even share.
# ============================================================

def exp4_persistence():
    c = E2_CERT[E2_CERT["game"].isin(CFG.REMOVALS)].copy()
    rows = []
    for (key, pop, q, pair), g in c.groupby(["head", "population", "quantity", "pair"]):
        conf = len(g)
        full = conf == len(CFG.SEEDS) * len(CFG.REMOVALS)
        pos = int((g["bernstein_sign"] > 0).sum())
        neg = int((g["bernstein_sign"] < 0).sum())
        rows.append({"head": key, "population": pop, "quantity": q, "pair": pair, "configurations": conf,
                     "certified_positive": pos, "certified_negative": neg,
                     "mean_over_configurations": float(g["mean"].mean()),
                     "min_mean": float(g["mean"].min()), "max_mean": float(g["mean"].max()),
                     "max_images_needed": float(g["images_needed_bernstein"].max()),
                     "persistent": ("positive" if full and pos == conf else "negative" if full and neg == conf else "")})
    return pd.DataFrame(rows)


banner("EXPERIMENT 4. PERSISTENCE ACROSS SEEDS AND REMOVAL RULES")
E4 = exp4_persistence()
show(E4, "4. Persistence of every certified population mean", key="e4_persistence", echo=False)
show(E4[E4["persistent"] != ""].sort_values(["head", "population", "quantity", "mean_over_configurations"]),
     "4. Findings that persist in every seed under both removal rules", key="e4_persistent")
show(E4[E4.quantity == "banzhaf_interaction"].groupby(["head", "population"]).agg(
    pairs=("pair", "size"), persistent=("persistent", lambda x: int((x != "").sum()))).reset_index(),
     "4. Persistent pairwise interactions per head and population", key="e4_counts")
_u = E4[(E4.quantity == "banzhaf_interaction") & (E4["persistent"] == "")]
_u = _u[(_u.certified_positive + _u.certified_negative) > 0]
show(_u.sort_values("max_images_needed").head(30),
     "4. Pairs certified in some but not all configurations, with the images the least powered one would need",
     key="e4_unresolved")

## Output files

In [ ]:
# ============================================================
# CSV FILES AND RUN SUMMARY
# One csv file per table type, written once. The zip archive holds the csv files, the console log and the
# frozen choices, which is everything needed to finalize the paper.
# ============================================================
import shutil

for key, frames in TABLES.items():
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(os.path.join(CFG.CSV_DIR, f"{key}.csv"), index=False)
say(f"\n{len(TABLES)} csv files written to {CFG.CSV_DIR}")
with open(os.path.join(CFG.OUTPUT_DIR, "run_info.json"), "w") as fh:
    json.dump({"seeds": CFG.SEEDS, "selected_seed": SEL, "beta": {str(k): v for k, v in BETA.items()},
               "vocabularies": VOCABS, "concept_names": CONCEPT_NAMES, "heads": HEAD_KEYS,
               "finished": time.strftime("%Y-%m-%d %H:%M:%S")}, fh, indent=1, default=str)
archive = shutil.make_archive(os.path.join(os.path.dirname(CFG.OUTPUT_DIR.rstrip("/")), "AIA_SkinCon_outputs"), "zip",
                              CFG.OUTPUT_DIR, ".")
say(f"Archive for the paper: {archive}")